# Vattenfall Silver Layer Business Intelligence

## Overview
This notebook answers key business questions using the Silver layer tables:
- `silver_grid_events` - Cleaned grid incident data
- `silver_asset_reference` - Integrated asset catalog
- `silver_regional_operations_base` - Pre-joined operational dataset

## Business Questions
1. Which regions show elevated operational incidents?
2. Which days show high market price pressure?
3. How do weather conditions relate to grid events?
4. Which regions need operational attention?
5. What summary tables can feed dashboards?

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, count, sum, avg, round, when, countDistinct, stddev, max, min, date_format, lower
from pyspark.sql.window import Window

# Load base tables - USE SILVER TABLES FOR INSIGHTS
df_operations = spark.table("vattenfall_dev.refined.silver_regional_operations_base")
df_market_prices = spark.table("vattenfall_dev.refined.silver_market_prices")  # Changed from bronze to silver
df_weather = spark.table("vattenfall_dev.refined.silver_weather")  # Changed from bronze to silver

print(f"✅ Loaded {df_operations.count()} operational records")
print(f"✅ Loaded {df_market_prices.count()} market price observations")
print(f"✅ Loaded {df_weather.count()} weather observations")

## Question 1: Which Regions Show Elevated Operational Incidents?

**Analysis Focus:**
- Total incident frequency by region
- Customer impact metrics
- Critical incident patterns
- High-risk event identification

In [0]:
# Question 1: Regional Incidents Analysis
regional_incidents = df_operations.groupBy("region_name", "country_name").agg(
    count("*").alias("total_incidents"),
    countDistinct("substation_id").alias("affected_substations"),
    sum("affected_customers").alias("total_customers_impacted"),
    round(avg("duration_minutes"), 1).alias("avg_duration_min"),
    round(avg("population_impact_rate"), 1).alias("avg_population_impact"),
    sum(when(col("severity") == "critical", 1).otherwise(0)).alias("critical_incidents"),
    sum(when(col("high_risk_event") == True, 1).otherwise(0)).alias("high_risk_events"),
    round(avg("impact_intensity"), 2).alias("avg_impact_intensity")
).orderBy(col("total_incidents").desc(), col("avg_population_impact").desc())

display(regional_incidents)

### Key Findings: Regional Incidents

**🔴 Critical Attention Required:**
- **Turku, Finland**: 3 incidents, 7,031 customers affected, **1,200 population impact rate** (worst)
  - 2 critical incidents with high-risk flags
  - Requires immediate capacity expansion
  
- **Copenhagen, Denmark**: 3 incidents, 6,022 customers affected
  - **228 min average duration** (restoration speed issues)
  - 1 high-risk event flagged

**🟠 Monitor Closely:**
- **Helsinki, Finland**: 2 incidents, 7,363 customers, 2 critical, 1 high-risk
- **Tampere, Finland**: 2 incidents, 3,273 customers, 670 population impact rate

**📊 Pattern:** Finland accounts for **58% of all incidents** (7 of 12), indicating systemic infrastructure challenges.

## Question 2: Which Days Show High Market Price Pressure?

**Analysis Focus:**
- Daily average energy prices
- Price volatility patterns
- Regional price differences
- High-price hour identification

In [0]:
# Question 2: Market Price Analysis
market_price_analysis = df_market_prices \
    .groupBy("report_day", "region_normalized").agg(
        count("*").alias("price_observations"),
        round(avg("price_eur_mwh"), 2).alias("avg_price_eur_mwh"),
        round(min("price_eur_mwh"), 2).alias("min_price_eur_mwh"),
        round(max("price_eur_mwh"), 2).alias("max_price_eur_mwh"),
        round(stddev("price_eur_mwh"), 2).alias("price_volatility"),
        sum(when(col("price_eur_mwh") > 70, 1).otherwise(0)).alias("high_price_hours")
    ) \
    .orderBy(col("avg_price_eur_mwh").desc(), col("price_volatility").desc()) \
    .limit(20)

display(market_price_analysis)

### Key Findings: Market Prices

**Highest Price Days (January 2024):**

| Date | Region | Avg Price | Min | Max | Volatility | Observations |
|------|--------|-----------|-----|-----|------------|-------------|
| **Jan 9** | **FI** | **56.66** | 47.23 | 66.38 | 8.18 | 7 |
| Jan 7 | NO | 55.27 | 39.35 | 66.49 | 10.37 | 7 |
| Jan 13 | FI | 53.89 | 44.30 | 61.12 | 6.56 | 5 |
| Jan 10 | FI | 52.90 | 51.84 | 54.41 | 1.34 | 3 |
| Jan 12 | FI | 52.32 | 24.17 | 64.28 | **14.55** | 6 |

**Regional Insights:**
- **Finland (FI)**: Highest average prices - **dominates top 5 days** (4 of 5)
  - Peak: 56.66 EUR/MWh (Jan 9)
  - Most volatile day: Jan 12 (14.55 stddev, 24.17-64.28 EUR/MWh range)
  
- **Norway (NO)**: Second highest prices
  - Jan 7: 55.27 EUR/MWh (10.37 volatility)
  
- **Sweden (SE)**: Moderate pricing
  - Jan 10: 50.86 EUR/MWh across 22 observations
  - Jan 15: 49.72 EUR/MWh across 22 observations
  
- **Denmark (DK)**: Competitive pricing
  - Jan 12: 49.33 EUR/MWh (15.73 volatility - highest for DK)
  - Jan 1: 49.09 EUR/MWh

**Price Characteristics:**
- **Average range**: 47-57 EUR/MWh (moderate)
- **Peak prices**: 61-69 EUR/MWh (no extreme spikes observed)
- **Volatility range**: 1.34-15.73 EUR/MWh standard deviation
  - Lowest volatility: Jan 10 FI (1.34) - very stable
  - Highest volatility: Jan 12 DK (15.73) and FI (14.55) - significant price swings
- **No extreme spikes**: All prices under 70 EUR/MWh threshold (0 high_price_hours recorded)

**📈 Key Patterns:**
1. **Finland price premium**: FI consistently pays 5-10% more than other regions
2. **Mid-month volatility**: Jan 12 shows high volatility across both FI and DK (14-15 stddev)
3. **Stable periods**: Jan 10 FI was exceptionally stable (1.34 volatility)
4. **No price spikes**: Market remained orderly throughout January - no correlation with grid incidents

**💡 Business Insight:**
- Market prices are **stable and moderate** during the observation period
- **No correlation** between high energy prices and grid operational incidents
- Finland's price premium suggests **demand pressure or supply constraints** - not grid-related
- Volatility on Jan 12 (DK: 15.73, FI: 14.55) warrants investigation - possible market event

## Question 3: How Do Weather Conditions Relate to Grid Events?

**Analysis Focus:**
- Temperature patterns during incidents
- Wind speed correlation
- Precipitation levels
- Critical event weather profiles

In [0]:
# Question 3: Weather-Event Correlation

# Step 1: Add city → country mapping to events
events_by_date = df_operations \
    .withColumn("event_date", F.to_date("timestamp")) \
    .withColumn(
        "country_code",
        F.when(F.col("region_name").isin("Helsinki", "Turku", "Tampere"), "FI")
         .when(F.col("region_name") == "Copenhagen", "DK")
         .when(F.col("region_name").isin("Gothenburg", "Malmo"), "SE")
         .otherwise(None)
    ) \
    .select("event_id", "event_date", "region_name", "country_code", "affected_customers", "severity")

# Step 2: Aggregate weather to DAILY level (avoid cartesian product with hourly data)
weather_daily = df_weather \
    .groupBy("report_day", F.lower(col("region_normalized")).alias("weather_country")).agg(
        round(avg("temperature_c"), 1).alias("avg_temp_c"),
        round(avg("wind_speed_ms"), 1).alias("avg_wind_speed_ms"),
        round(avg("precipitation_mm"), 2).alias("avg_precipitation_mm"),
        round(avg("cloud_cover_percent"), 1).alias("avg_cloud_cover_pct")
    )

# Step 3: Join on date AND country code (now 1:1 join, no cartesian product)
weather_correlation = events_by_date \
    .join(
        weather_daily,
        (events_by_date.event_date == weather_daily.report_day) &
        (lower(events_by_date.country_code) == weather_daily.weather_country),
        "left"
    ) \
    .groupBy("event_date", "region_name", "country_code").agg(
        countDistinct("event_id").alias("total_events"),
        sum("affected_customers").alias("total_customers_affected"),
        round(avg("avg_temp_c"), 1).alias("avg_temp_c"),
        round(avg("avg_wind_speed_ms"), 1).alias("avg_wind_speed_ms"),
        round(avg("avg_precipitation_mm"), 2).alias("avg_precipitation_mm"),
        round(avg("avg_cloud_cover_pct"), 1).alias("avg_cloud_cover_pct"),
        sum(when(col("severity") == "critical", 1).otherwise(0)).alias("critical_events")
    ) \
    .orderBy(col("total_events").desc(), col("critical_events").desc()) \
    .limit(20)

print("✅ City-to-country mapping applied:")
print("   Helsinki, Turku, Tampere → FI")
print("   Copenhagen → DK")
print("   Gothenburg, Malmo → SE")
print("\n✅ Weather aggregated to daily level (prevents cartesian product)")
print("\n📊 Join now uses country codes - customer counts are accurate\n")

display(weather_correlation)

### Key Findings: Weather-Grid Correlation

**✅ WEATHER DATA SUCCESSFULLY INTEGRATED** (via city→country mapping)

**Top Weather-Event Correlations:**

| Date | City | Country | Events | Customers | Critical | Temp (°C) | Wind (m/s) | Precip (mm) | Cloud (%) |
|------|------|---------|--------|-----------|----------|-----------|------------|-------------|----------|
| **Jan 4** | **Helsinki** | **FI** | **2** | **7,363** | **2** | **-2.4** | 10.5 | 2.05 | 49.2 |
| Jan 9 | Turku | FI | 2 | 4,829 | 1 | -1.4 | 10.2 | 2.04 | 54.1 |
| Jan 6 | Gothenburg | SE | 1 | 2,601 | 1 | -2.5 | 10.2 | 2.17 | 48.9 |
| Jan 2 | Copenhagen | DK | 1 | 4,688 | 1 | -2.8 | 10.2 | 3.12 | 43.2 |
| Jan 1 | Turku | FI | 1 | 2,202 | 1 | **-4.1** | 10.5 | **2.93** | 51.9 |
| Jan 5 | Tampere | FI | 1 | 1,784 | 0 | -3.4 | **8.2** | 2.69 | 51.3 |

**🌡️ Temperature Patterns:**
- **Range**: -4.1°C (Jan 1 Turku) to -1.4°C (Jan 9 Turku)
- **Average during incidents**: **-2.8°C** (below freezing)
- **Coldest event**: Jan 1 Turku at -4.1°C (1 critical incident)
- **Warmest event**: Jan 9 Turku at -1.4°C (1 critical incident)
- **Finding**: No clear correlation between extreme cold and incident severity

**💨 Wind Speed Patterns:**
- **Range**: 8.2 m/s (Jan 5 Tampere) to 10.5 m/s (Jan 4 Helsinki, Jan 1 Turku, Jan 7 Copenhagen)
- **Average during incidents**: **9.9 m/s** (fresh breeze, 20 mph)
- **Highest wind events**: 
  - Jan 4 Helsinki: 10.5 m/s → 2 critical events, 7,363 customers
  - Jan 7 Copenhagen: 10.5 m/s → 0 critical events, 594 customers
- **Finding**: Wind speeds are consistently elevated (8-10.5 m/s) across ALL incidents

**🌧️ Precipitation Patterns:**
- **Range**: 2.04 mm (Jan 9 Turku) to 3.12 mm (Jan 2 Copenhagen)
- **Average during incidents**: **2.48 mm** (light rain/snow)
- **Highest precipitation**: Jan 2 Copenhagen (3.12 mm) → 1 critical, 4,688 customers
- **Finding**: Moderate precipitation present during all incidents (winter rain/snow mix)

**☁️ Cloud Cover:**
- **Range**: 43.2% (Jan 2 Copenhagen) to 54.5% (Jan 6 Copenhagen)
- **Average**: **50%** (partly cloudy to mostly cloudy)
- **Finding**: Consistent cloud cover, no clear pattern with incident severity

---

**📊 KEY INSIGHTS:**

1. **Wind is the strongest correlate**: ALL incidents occur with wind speeds 8-10.5 m/s
   - **Hypothesis**: Nordic grid infrastructure is vulnerable to sustained wind loads
   - **Action**: Review wind-resistance specifications for aging assets

2. **Cold + Wind + Precipitation = Triple Threat**:
   - Jan 4 Helsinki (WORST DAY): -2.4°C + 10.5 m/s wind + 2.05 mm precip → **2 critical, 7,363 customers**
   - Jan 1 Turku: -4.1°C + 10.5 m/s wind + 2.93 mm precip → 1 critical, 2,202 customers
   - **Pattern**: Sub-zero temps + high wind + precipitation = elevated risk

3. **Temperature alone is NOT predictive**:
   - Coldest event (-4.1°C, Jan 1): Only 1 critical incident
   - Warmer event (-1.4°C, Jan 9): Also 1 critical incident
   - **Conclusion**: Temperature severity doesn't directly correlate with incident severity

4. **Country-Level Weather Limitation**:
   - Weather data is country-aggregated (FI, DK, SE) while events are city-specific
   - Local microclimates (coastal vs inland) are not captured
   - **Gold layer enhancement**: Add city-level weather stations for precise correlation

---

**🚨 PREDICTIVE WEATHER ALERTING RECOMMENDATIONS:**

1. **High-Risk Weather Profile**:
   - Temperature: Below -2°C
   - Wind speed: Above 10 m/s
   - Precipitation: Above 2 mm
   - → **Pre-position maintenance crews in affected regions**

2. **Focus Regions for Weather Monitoring**:
   - **Finland (FI)**: 7 of 10 events, most sensitive to weather conditions
   - Monitor Helsinki, Turku, Tampere during winter storms

3. **Gold Layer Enhancement**:
   - Create `gold_weather_enhanced_events` with:
     - City-level weather mapping
     - Weather severity scores
     - Predictive risk flags (cold + wind + precip)
   - Build ML model: Predict incident probability from weather forecasts

**💡 Business Value**:
- **Proactive crew deployment** during high-risk weather (reduce restoration time)
- **Customer communication** ahead of weather events (manage expectations)
- **Infrastructure hardening** priority based on weather vulnerability analysis

## Question 4: Which Regions Need Operational Attention?

**Analysis Focus:**
- Composite risk scoring
- Aging infrastructure identification
- Restoration time performance
- Population impact severity

In [0]:
# Question 4: Composite Risk Scoring
composite_risk = df_operations.groupBy("region_name").agg(
    countDistinct("substation_id").alias("total_substations"),
    sum(when(col("asset_age_category") == "aging", 1).otherwise(0)).alias("aging_substations"),
    sum(when(col("high_risk_event") == True, 1).otherwise(0)).alias("high_risk_events"),
    round(avg("asset_age_years"), 1).alias("avg_asset_age_years"),
    round(sum("affected_customers") / 1000.0, 1).alias("total_customers_affected_k"),
    round(avg("duration_minutes"), 1).alias("avg_duration_min")
).withColumn(
    "composite_risk_score",
    round(
        (col("aging_substations") * 2) +
        (col("high_risk_events") * 3) +
        (col("avg_asset_age_years") * 0.5) +
        (col("avg_duration_min") * 0.1),
        1
    )
).orderBy(col("composite_risk_score").desc())

display(composite_risk)

### Key Findings: Operational Priority Ranking

**🔴 Risk Score Rankings (Higher = More Urgent):**

| Rank | Region | Risk Score | Substations | Aging Assets | High-Risk Events | Avg Duration |
|------|--------|------------|-------------|--------------|------------------|-------------|
| 1 | **Copenhagen** | **43.0** | 3 | 2 | 1 | **228 min** |
| 2 | **Turku** | **22.6** | 3 | 0 | 2 | 70 min |
| 3 | **Gothenburg** | **22.5** | 1 | 0 | 0 | 175 min |
| 4 | Helsinki | 19.0 | 2 | 0 | 1 | 72 min |
| 5 | Malmo | 15.0 | 1 | 0 | 0 | 85 min |
| 6 | Tampere | 15.0 | 1 | 0 | 0 | 85 min |

**⚠️ Immediate Actions Required:**

1. **Copenhagen** (Risk: **43.0** - CRITICAL)
   - **Primary Issue**: 228 min average restoration time (3.8 hours!)
   - **Root Cause**: SUB128 (31 yrs aging) had **383 min outage** (6.4 hours)
   - **Action**: Replace SUB128 immediately, retrain restoration crews
   - **Impact**: 2 aging substations (SUB128, SUB130) serving 6,022 customers

2. **Turku** (Risk: 22.6)
   - **2 high-risk events** recorded
   - 7,031 customers affected across 3 substations
   - Includes SUB136 (28 yrs mature) and SUB105 (26 yrs mature)
   - **Action**: Comprehensive infrastructure audit

3. **Gothenburg** (Risk: 22.5) 
   - **Unexpected**: Only 1 substation, modern (10 yrs), but **175 min** restoration time
   - SUB121 serves 2,601 customers with high voltage needs
   - **Action**: Investigate why restoration takes so long despite modern asset

**💡 Key Insight**: Copenhagen's risk is **DOUBLE** any other region due to catastrophic restoration times, not just asset age.

## Question 5: What Summary Tables Can Feed Dashboards?

**Dashboard Types:**
- **5a. Daily Operational KPIs** - Executive daily briefing
- **5b. Asset Reliability** - Maintenance prioritization
- **5c. Regional Performance** - Strategic planning scorecard

In [0]:
# 5a. Daily Operational KPIs
daily_kpis = df_operations \
    .withColumn("operational_date", F.to_date("timestamp")) \
    .groupBy("operational_date", "country_name").agg(
        countDistinct("event_id").alias("daily_incidents"),
        countDistinct("substation_id").alias("affected_assets"),
        sum("affected_customers").alias("total_customers_affected"),
        round(avg("duration_minutes"), 1).alias("avg_restoration_min"),
        sum(when(col("severity") == "critical", 1).otherwise(0)).alias("critical_count"),
        sum(when(col("severity") == "major", 1).otherwise(0)).alias("major_count"),
        sum(when(col("severity") == "moderate", 1).otherwise(0)).alias("moderate_count"),
        sum(when(col("high_risk_event") == True, 1).otherwise(0)).alias("high_risk_count"),
        round(avg("impact_intensity"), 2).alias("avg_impact_intensity")
    ) \
    .orderBy(col("operational_date").desc(), col("country_name"))

display(daily_kpis)

In [0]:
# 5b. Asset Reliability Scoring
asset_reliability = df_operations.groupBy(
    "substation_id", "region_name", "country_name", "capacity_mva",
    "voltage_level", "asset_age_years", "asset_age_category"
).agg(
    count("*").alias("incident_count"),
    sum("affected_customers").alias("total_customers_impacted"),
    round(avg("duration_minutes"), 1).alias("avg_outage_duration"),
    round(avg("impact_intensity"), 2).alias("avg_impact_intensity"),
    sum(when(col("severity") == "critical", 1).otherwise(0)).alias("critical_incidents"),
    max(when(col("high_risk_event") == True, 1).otherwise(0)).alias("has_high_risk_event")
).withColumn(
    "reliability_score",
    round(F.lit(100.0) / (F.lit(1.0) + col("incident_count") + col("avg_impact_intensity")), 2)
).orderBy(col("reliability_score").asc(), col("incident_count").desc())

display(asset_reliability)

In [0]:
# 5c. Regional Performance Scorecard
regional_scorecard = df_operations.groupBy("region_name", "country_name").agg(
    count("*").alias("total_incidents"),
    sum("affected_customers").alias("total_customers_affected"),
    round(avg("duration_minutes"), 1).alias("avg_duration_min"),
    round(avg("population_impact_rate"), 1).alias("avg_pop_impact_rate"),
    countDistinct("substation_id").alias("total_substations"),
    sum(when(col("severity") == "critical", 1).otherwise(0)).alias("critical_incidents"),
    sum(when(col("high_risk_event") == True, 1).otherwise(0)).alias("high_risk_events")
).withColumn(
    "performance_grade",
    when((col("total_incidents") >= 5) & (col("avg_pop_impact_rate") > 500), "D - Critical Attention")
    .when((col("total_incidents") >= 3) & (col("avg_pop_impact_rate") > 300), "C - Needs Improvement")
    .when((col("total_incidents") >= 2) | (col("avg_pop_impact_rate") > 200), "B - Monitor Closely")
    .otherwise("A - Good Performance")
).withColumn(
    "grade_rank",
    when((col("total_incidents") >= 5) & (col("avg_pop_impact_rate") > 500), 1)
    .when((col("total_incidents") >= 3) & (col("avg_pop_impact_rate") > 300), 2)
    .when((col("total_incidents") >= 2) | (col("avg_pop_impact_rate") > 200), 3)
    .otherwise(4)
).orderBy(col("grade_rank"), col("total_incidents").desc()).drop("grade_rank")

display(regional_scorecard)

### Key Findings: Dashboard Summary

**📊 Dashboard Highlights:**

1. **Daily Operational KPIs** (5a) - Concrete Insights:
   
   **Worst Days:**
   - **Jan 4 (Finland)**: 2 incidents, **7,363 customers** (highest impact), 2 critical events, 1 high-risk
   - **Jan 2 (Denmark)**: **383 min restoration time** (6.4 hours - completely unacceptable)
   - **Jan 9 (Finland)**: 2 incidents, 4,829 customers, 1 high-risk event
   
   **Country Performance:**
   - Finland: Incidents on **7 of 9 days** tracked (Jan 1, 4, 5, 7, 9)
   - Denmark: 3 incident days (Jan 2, 6, 7) - restoration speed crisis
   - Sweden: 2 incident days (Jan 2, 6) - best performance
   
   **Critical Pattern**: Finland experiences incidents almost daily, Denmark has severe restoration time issues.

2. **Asset Reliability Scoring** (5b) - **UPDATED WITH ACTUAL DATA**:
   
   **Bottom 3 Assets (Lowest Reliability):**
   - **SUB128** (Copenhagen): **2.05 reliability**, 31 yrs (AGING), **383 min outage**, 4,688 customers - **IMMEDIATE REPLACEMENT**
   - **SUB149** (Helsinki): **2.50 reliability**, 23 yrs (mature), 3,793 customers, **37.93 impact intensity** - CRITICAL
   - **SUB112** (Helsinki): **2.65 reliability**, 12 yrs (modern but failing!), 3,570 customers - INVESTIGATE ROOT CAUSE
   
   **Key Issue**: SUB128 has the **worst restoration time ever recorded** (6+ hours). Even modern assets like SUB112 are failing.

3. **Regional Performance Scorecard** (5c):
   - **Grade C**: Turku (needs improvement)
   - **Grade B**: All other regions (monitor closely)
   - No regions at Grade A (performance gap)
   - No Grade D (no critical crisis state)


---

# 🎯 Key Grid Insights: Answering Critical Questions

## Questions We'll Answer:

1. **Why does Copenhagen have the highest risk score?** (And what is it composed of?)
2. **Why are SUB128, SUB149, and SUB112 the worst performers?** (And why is SUB121 bad despite being only 10 years old?)
3. **How do weather conditions affect grid events?**
4. **Are weather conditions the only direct cause?**
5. **What are the operational ratings for each country?**

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from matplotlib.patches import Rectangle

# Professional styling
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.titleweight'] = 'bold'

print("✅ Visualization setup complete")

## 🔴 Question 1: Why Does Copenhagen Have the Highest Risk Score?

**Risk Score Formula:**
```
Risk Score = (Aging Assets × 2) + (High-Risk Events × 3) + (Avg Asset Age × 0.5) + (Avg Duration × 0.1)
```

**What we'll discover:** Which component contributes most to Copenhagen's risk?

In [0]:
# Calculate risk score components for all regions
from pyspark.sql.functions import col, when, sum as _sum, avg as _avg, round as _round, countDistinct

risk_components = df_operations.groupBy("region_name").agg(
    countDistinct("substation_id").alias("total_substations"),
    _sum(when(col("asset_age_category") == "aging", 1).otherwise(0)).alias("aging_substations"),
    _sum(when(col("high_risk_event") == True, 1).otherwise(0)).alias("high_risk_events"),
    _round(_avg("asset_age_years"), 2).alias("avg_asset_age_years"),
    _round(_avg("duration_minutes"), 1).alias("avg_duration_min")
).toPandas()

# Calculate each component's contribution
risk_components['aging_score'] = risk_components['aging_substations'] * 2
risk_components['risk_events_score'] = risk_components['high_risk_events'] * 3
risk_components['age_score'] = risk_components['avg_asset_age_years'] * 0.5
risk_components['duration_score'] = risk_components['avg_duration_min'] * 0.1
risk_components['total_risk_score'] = (
    risk_components['aging_score'] + 
    risk_components['risk_events_score'] + 
    risk_components['age_score'] + 
    risk_components['duration_score']
).round(1)

# Sort by total risk score descending
risk_components = risk_components.sort_values('total_risk_score', ascending=False)

# Create stacked bar chart
fig, ax = plt.subplots(figsize=(14, 8))

x = np.arange(len(risk_components))
width = 0.6

# Stack the components
p1 = ax.bar(x, risk_components['aging_score'], width, label='Aging Assets (×2)', color='#E67E22')
p2 = ax.bar(x, risk_components['risk_events_score'], width, bottom=risk_components['aging_score'],
            label='High-Risk Events (×3)', color='#E74C3C')
p3 = ax.bar(x, risk_components['age_score'], width, 
            bottom=risk_components['aging_score'] + risk_components['risk_events_score'],
            label='Average Age (×0.5)', color='#F39C12')
p4 = ax.bar(x, risk_components['duration_score'], width,
            bottom=risk_components['aging_score'] + risk_components['risk_events_score'] + risk_components['age_score'],
            label='Restoration Duration (×0.1)', color='#3498DB')

# Add total score labels on top
for i, (idx, row) in enumerate(risk_components.iterrows()):
    total = row['total_risk_score']
    ax.text(i, total + 1.5, f"{total}", ha='center', va='bottom', fontsize=13, fontweight='bold')

ax.set_ylabel('Risk Score Points', fontsize=13, fontweight='bold')
ax.set_xlabel('Region', fontsize=13, fontweight='bold')
ax.set_title('Risk Score Composition by Region: Copenhagen\'s Duration Problem', 
             fontsize=16, fontweight='bold', pad=20)
ax.set_xticks(x)
ax.set_xticklabels(risk_components['region_name'], fontsize=12)
ax.legend(loc='upper right', framealpha=0.95, fontsize=11)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Print detailed breakdown
print("\n" + "="*80)
print("RISK SCORE BREAKDOWN BY REGION")
print("="*80)
for idx, row in risk_components.iterrows():
    print(f"\n{row['region_name']}:")
    print(f"  Total Risk Score: {row['total_risk_score']}")
    print(f"  - Aging Assets: {row['aging_substations']} substations × 2 = {row['aging_score']} pts ({row['aging_score']/row['total_risk_score']*100:.1f}%)")
    print(f"  - High-Risk Events: {row['high_risk_events']} events × 3 = {row['risk_events_score']} pts ({row['risk_events_score']/row['total_risk_score']*100:.1f}%)")
    print(f"  - Avg Asset Age: {row['avg_asset_age_years']} years × 0.5 = {row['age_score']:.1f} pts ({row['age_score']/row['total_risk_score']*100:.1f}%)")
    print(f"  - Avg Duration: {row['avg_duration_min']} min × 0.1 = {row['duration_score']:.1f} pts ({row['duration_score']/row['total_risk_score']*100:.1f}%)")

print("\n" + "="*80)
print("🎯 KEY FINDING:")
cph = risk_components[risk_components['region_name'] == 'Copenhagen'].iloc[0]
print(f"Copenhagen's restoration duration ({cph['avg_duration_min']} min) contributes")
print(f"{cph['duration_score']:.1f} points = {cph['duration_score']/cph['total_risk_score']*100:.1f}% of its total risk score!")
print("="*80)

## ⚠️ Question 2: Why Are SUB128, SUB149, and SUB112 the Worst Performers?

**Performance Score Factors:**
- Incident frequency
- Restoration time
- Customer impact
- Asset age
- Critical incidents

**Special Investigation:** Why is SUB121 a bad performer despite being only 10 years old?

In [0]:
# Analyze all substations
substation_analysis = df_operations.groupBy(
    "substation_id", "region_name", "asset_age_years", "asset_age_category"
).agg(
    F.count("*").alias("incident_count"),
    _sum("affected_customers").alias("total_customers"),
    _round(_avg("duration_minutes"), 1).alias("avg_duration_min"),
    _sum(when(col("severity") == "critical", 1).otherwise(0)).alias("critical_incidents"),
    F.max(when(col("high_risk_event") == True, 1).otherwise(0)).alias("has_high_risk")
).withColumn(
    "performance_score",
    _round(
        (col("incident_count") * 10) + 
        (col("avg_duration_min") * 0.5) + 
        (col("total_customers") / 100) + 
        (col("critical_incidents") * 15) +
        (col("has_high_risk") * 20) +
        (col("asset_age_years") * 0.5),
        1
    )
).orderBy(col("performance_score").desc()).toPandas()

# Get top 10 worst performers
top10_worst = substation_analysis.head(10).copy()

# Identify the key substations
worst_3 = ['SUB128', 'SUB149', 'SUB112']
sub121 = 'SUB121'

# Create clearer 2-panel visualization
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Define colors for regions
region_colors = {
    'Copenhagen': '#E74C3C',
    'Helsinki': '#E67E22', 
    'Turku': '#F39C12',
    'Gothenburg': '#3498DB',
    'Tampere': '#9B59B6',
    'Malmo': '#27AE60'
}

# Panel 1: Bar chart of Top 10 Worst Performers
ax1 = axes[0]
y_pos = np.arange(len(top10_worst))
colors = [region_colors.get(region, '#95A5A6') for region in top10_worst['region_name']]

bars = ax1.barh(y_pos, top10_worst['performance_score'], color=colors, edgecolor='black', linewidth=2)
ax1.set_yticks(y_pos)
ax1.set_yticklabels([f"{row['substation_id']} ({row['region_name']})" 
                      for _, row in top10_worst.iterrows()], fontsize=11)
ax1.invert_yaxis()
ax1.set_xlabel('Performance Score (higher = worse)', fontsize=13, fontweight='bold')
ax1.set_title('Top 10 Worst Performing Substations', fontsize=15, fontweight='bold', pad=15)
ax1.grid(axis='x', alpha=0.3)

# Add value labels
for i, (bar, score) in enumerate(zip(bars, top10_worst['performance_score'])):
    ax1.text(score + 5, bar.get_y() + bar.get_height()/2, f'{score:.1f}',
             va='center', fontsize=11, fontweight='bold')

# Highlight special substations with markers
for i, (_, row) in enumerate(top10_worst.iterrows()):
    if row['substation_id'] in worst_3:
        ax1.text(5, i, '⚠️', fontsize=16, ha='left', va='center')
    elif row['substation_id'] == sub121:
        ax1.text(5, i, '💎', fontsize=16, ha='left', va='center')

# Add legend for markers
ax1.text(0.02, 0.98, '⚠️ = User-specified worst performers\n💎 = SUB121 (Modern but poor performance)', 
         transform=ax1.transAxes, fontsize=10, verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='#FFF9E6', alpha=0.9, edgecolor='black', linewidth=1))

# Panel 2: Age Analysis - Focus on the Paradox
ax2 = axes[1]

# Scatter plot with age on x-axis, score on y-axis for top 10
for _, row in top10_worst.iterrows():
    sub_id = row['substation_id']
    
    # Special highlighting
    if sub_id == sub121:
        marker = 'D'
        size = 500
        edgecolor = 'black'
        linewidth = 3
        alpha = 1.0
        zorder = 10
    elif sub_id in worst_3:
        marker = 'X'
        size = 400
        edgecolor = 'black'
        linewidth = 2.5
        alpha = 1.0
        zorder = 9
    else:
        marker = 'o'
        size = 250
        edgecolor = 'gray'
        linewidth = 1.5
        alpha = 0.7
        zorder = 5
    
    ax2.scatter(row['asset_age_years'], row['performance_score'], 
               s=size, color=region_colors.get(row['region_name'], '#95A5A6'),
               marker=marker, alpha=alpha, edgecolors=edgecolor, linewidth=linewidth, zorder=zorder)
    
    # Label each substation
    ax2.text(row['asset_age_years'], row['performance_score'] + 8, 
            f"{sub_id}\n{int(row['asset_age_years'])} yrs",
            fontsize=9, fontweight='bold', ha='center')

ax2.set_xlabel('Asset Age (years)', fontsize=13, fontweight='bold')
ax2.set_ylabel('Performance Score (higher = worse)', fontsize=13, fontweight='bold')
ax2.set_title('Age vs Performance: The SUB121 Anomaly', fontsize=15, fontweight='bold', pad=15)
ax2.grid(True, alpha=0.3)

# Add vertical line at 15 years (aging threshold)
ax2.axvline(x=20, color='red', linestyle='--', linewidth=2, alpha=0.5, label='Typical aging threshold')
ax2.legend(loc='upper left', fontsize=10)

# Highlight SUB121 paradox with annotation
sub121_row = top10_worst[top10_worst['substation_id'] == sub121]
if not sub121_row.empty:
    row = sub121_row.iloc[0]
    ax2.annotate('SUB121: Only 10 years old\nbut 2nd worst performer!\n(Operational issue)', 
                xy=(row['asset_age_years'], row['performance_score']),
                xytext=(row['asset_age_years'] - 8, row['performance_score'] + 50),
                fontsize=11, fontweight='bold', color='#3498DB',
                bbox=dict(boxstyle='round', facecolor='#E3F2FD', alpha=0.95, edgecolor='#3498DB', linewidth=2),
                arrowprops=dict(arrowstyle='->', lw=2, color='#3498DB'))

plt.suptitle('Worst Performing Substations: SUB128 vs SUB121 Paradox', 
             fontsize=17, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

# Print analysis of key substations
print("\n" + "="*80)
print("TOP 10 WORST PERFORMING SUBSTATIONS")
print("="*80)
print(f"\n{'Rank':<6} {'Substation':<12} {'Region':<12} {'Score':<8} {'Age':<6} {'Rest.Time':<12} {'Customers':<10}")
print("-" * 80)

for idx, (_, row) in enumerate(top10_worst.iterrows(), 1):
    marker = "⚠️" if row['substation_id'] in worst_3 else "💎" if row['substation_id'] == sub121 else "  "
    print(f"{marker} #{idx:<4} {row['substation_id']:<12} {row['region_name']:<12} "
          f"{row['performance_score']:<8.1f} {int(row['asset_age_years']):<6} "
          f"{row['avg_duration_min']:<12.0f} {int(row['total_customers']):<10,}")

print("\n" + "="*80)
print("🎯 KEY FINDINGS")
print("="*80)

print("\n1. SUB128 (Copenhagen) - THE WORST:")
sub128 = top10_worst[top10_worst['substation_id'] == 'SUB128']
if not sub128.empty:
    s = sub128.iloc[0]
    print(f"   - Score: {s['performance_score']:.1f} (Rank #1)")
    print(f"   - Age: {int(s['asset_age_years'])} years (aging asset)")
    print(f"   - Restoration: {s['avg_duration_min']:.0f} minutes (catastrophic)")
    print(f"   - Customers: {int(s['total_customers']):,}")
    print(f"   → Both aging infrastructure AND operational problems")

print("\n2. SUB121 (Gothenburg) - THE PARADOX:")
sub121_data = top10_worst[top10_worst['substation_id'] == sub121]
if not sub121_data.empty:
    s = sub121_data.iloc[0]
    rank = top10_worst[top10_worst['substation_id'] == sub121].index[0] + 1
    print(f"   - Score: {s['performance_score']:.1f} (Rank #{rank})")
    print(f"   - Age: {int(s['asset_age_years'])} years (MODERN ASSET!)")
    print(f"   - Restoration: {s['avg_duration_min']:.0f} minutes")
    print(f"   - Customers: {int(s['total_customers']):,}")
    print(f"   → This is 100% OPERATIONAL FAILURE, not asset age!")
    print(f"   → Despite being modern, performs WORSE than older assets")

print("\n3. SUB149 & SUB112 (Helsinki):")
for sub_id in ['SUB149', 'SUB112']:
    sub_data = top10_worst[top10_worst['substation_id'] == sub_id]
    if not sub_data.empty:
        s = sub_data.iloc[0]
        rank = top10_worst[top10_worst['substation_id'] == sub_id].index[0] + 1
        print(f"   - {sub_id}: Score {s['performance_score']:.1f} (Rank #{rank}), "
              f"{int(s['asset_age_years'])} yrs, {s['avg_duration_min']:.0f} min restoration")

print("\n4. THE BIG LESSON:")
print("   Age is NOT the primary predictor of poor performance!")
print("   Modern equipment with poor operations > Old equipment with good operations")
print("   → Fix operational issues (training, protocols) BEFORE replacing assets")
print("="*80)

## 🔍 Question 3: Are Weather Conditions the Only Direct Cause?

**Investigation:**
- Weather-only incidents (modern assets)
- Weather + Aging asset incidents  
- Other factors

**What we'll discover:** Is weather sufficient, or do other factors amplify the impact?

In [0]:
# Categorize incidents by causation
incidents_categorized = incidents_weather.withColumn(
    "has_adverse_weather",
    when((col("is_cold") == 1) | (col("is_high_wind") == 1) | (col("has_precipitation") == 1), 1).otherwise(0)
).withColumn(
    "is_aging_asset",
    when(col("asset_age_category") == "aging", 1).otherwise(0)
).withColumn(
    "cause_category",
    when((col("has_adverse_weather") == 1) & (col("is_aging_asset") == 1), "Weather + Aging Asset")
    .when(col("has_adverse_weather") == 1, "Weather Only")
    .when(col("is_aging_asset") == 1, "Aging Asset Only")
    .otherwise("Other Factors")
)

# Aggregate by cause
cause_summary = incidents_categorized.groupBy("cause_category").agg(
    F.count("*").alias("incident_count"),
    _round(_avg("duration_minutes"), 1).alias("avg_restoration_min"),
    _round(_avg("affected_customers"), 0).alias("avg_customers"),
    _sum("affected_customers").alias("total_customers")
).orderBy(col("incident_count").desc()).toPandas()

# Create visualizations with CHEERFUL COLORS
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Panel 1: Pie chart of incident distribution
ax1 = axes[0]
colors_pie = ['#FFD93D', '#FF9FF3', '#B4E7CE', '#4ECDC4']
explode = [0.05 if cat in ['Weather Only', 'Weather + Aging Asset'] else 0 for cat in cause_summary['cause_category']]

wedges, texts, autotexts = ax1.pie(cause_summary['incident_count'], 
                                     labels=cause_summary['cause_category'],
                                     autopct='%1.0f%%',
                                     colors=colors_pie,
                                     explode=explode,
                                     startangle=90,
                                     textprops={'fontsize': 12, 'fontweight': 'bold'})

# Make percentage text larger and white
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontsize(14)
    autotext.set_fontweight('bold')

ax1.set_title('Incident Distribution by Cause Category', fontsize=14, fontweight='bold', pad=20)

# Panel 2: Bar chart comparing severity (restoration time)
ax2 = axes[1]
x_pos = np.arange(len(cause_summary))
colors_bar = ['#FFD93D', '#FF9FF3', '#B4E7CE', '#4ECDC4'][:len(cause_summary)]

bars = ax2.bar(x_pos, cause_summary['avg_restoration_min'], color=colors_bar, edgecolor='white', linewidth=2)
ax2.set_ylabel('Average Restoration Time (minutes)', fontsize=13, fontweight='bold')
ax2.set_xlabel('Cause Category', fontsize=13, fontweight='bold')
ax2.set_title('Severity by Cause: Which Failures Take Longer?', fontsize=14, fontweight='bold')
ax2.set_xticks(x_pos)
ax2.set_xticklabels(cause_summary['cause_category'], rotation=15, ha='right')
ax2.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar, val in zip(bars, cause_summary['avg_restoration_min']):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height + 5,
             f'{val:.0f} min',
             ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.suptitle('Incident Causation Analysis: What Drives Grid Failures?', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("INCIDENT CAUSATION BREAKDOWN")
print("="*80)

for idx, row in cause_summary.iterrows():
    pct = (row['incident_count'] / cause_summary['incident_count'].sum()) * 100
    print(f"\n{row['cause_category']}:")
    print(f"  - Incidents: {row['incident_count']} ({pct:.0f}%)")
    print(f"  - Avg restoration: {row['avg_restoration_min']:.0f} min")
    print(f"  - Avg customers: {int(row['avg_customers']):,}")
    print(f"  - Total customers: {int(row['total_customers']):,}")

print("\n" + "="*80)
print("🎯 KEY INSIGHTS:")
print("="*80)

weather_incidents = cause_summary[cause_summary['cause_category'].str.contains('Weather')]['incident_count'].sum()
total = cause_summary['incident_count'].sum()
weather_pct = (weather_incidents / total) * 100
print(f"\n• {weather_pct:.0f}% of incidents involve adverse weather")

aging_incidents = cause_summary[cause_summary['cause_category'].str.contains('Aging')]['incident_count'].sum()
aging_pct = (aging_incidents / total) * 100
print(f"• {aging_pct:.0f}% involve aging infrastructure")

combined = cause_summary[cause_summary['cause_category'] == 'Weather + Aging Asset']['incident_count'].sum() if 'Weather + Aging Asset' in cause_summary['cause_category'].values else 0
if combined > 0:
    print(f"• {combined} incidents show COMPOUNDING RISK (weather + aging)")

print("="*80)

## 🏏 Question 5: What Are the Operational Ratings for Each Country?

**Rating Scale:**
- ⭐ **EXCELLENT** (80-100): Optimal conditions
- 🟢 **GOOD** (60-80): Low incident activity  
- 🟡 **FAIR** (40-60): Moderate stress
- 🟠 **POOR** (20-40): Elevated incidents
- 🔴 **CRITICAL** (0-20): Major incidents

**What we'll discover:** How does each country perform overall?

In [0]:
# Load daily ratings from gold layer
daily_ratings = spark.table("vattenfall_dev.gold.gold_regional_condition_daily").select(
    "report_day", "region", "operational_condition", "health_score", 
    "total_incident_count", "total_customers_affected"
).toPandas()

# Map region codes to country names
region_to_country = {
    'DK': 'Denmark',
    'FI': 'Finland',
    'SE': 'Sweden',
    'NO': 'Norway'
}

daily_ratings['country'] = daily_ratings['region'].map(region_to_country)

# Calculate country-level metrics
country_summary = daily_ratings.groupby('country').agg({
    'health_score': 'mean',
    'total_incident_count': 'sum',
    'total_customers_affected': 'sum',
    'report_day': 'count'
}).reset_index()

country_summary.columns = ['country', 'avg_health_score', 'total_incidents', 'total_customers', 'num_region_days']
country_summary = country_summary.sort_values('avg_health_score', ascending=False)

# Count rating distribution by country
rating_dist = daily_ratings.groupby(['country', 'operational_condition']).size().unstack(fill_value=0)
rating_order = ['EXCELLENT', 'GOOD', 'FAIR', 'POOR', 'CRITICAL']
rating_dist = rating_dist.reindex(columns=rating_order, fill_value=0)

# Create visualizations with CHEERFUL COLORS
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Panel 1: Average Health Score by Country
ax1 = axes[0, 0]
colors_score = ['#6BCB77' if score >= 60 else '#FFD93D' if score >= 40 else '#FFA07A' if score >= 20 else '#FF9FF3' 
                for score in country_summary['avg_health_score']]

bars1 = ax1.bar(country_summary['country'], country_summary['avg_health_score'], 
                color=colors_score, edgecolor='white', linewidth=2)
ax1.axhline(y=60, color='#6BCB77', linestyle='--', linewidth=2, alpha=0.5, label='GOOD threshold')
ax1.axhline(y=40, color='#FFD93D', linestyle='--', linewidth=2, alpha=0.5, label='FAIR threshold')
ax1.axhline(y=20, color='#FF9FF3', linestyle='--', linewidth=2, alpha=0.5, label='CRITICAL threshold')
ax1.set_ylabel('Average Health Score', fontsize=12, fontweight='bold')
ax1.set_title('Average Health Score by Country', fontsize=13, fontweight='bold')
ax1.set_ylim(0, 80)
ax1.legend(loc='upper right', fontsize=9)
ax1.grid(axis='y', alpha=0.3)

# Add value labels
for bar, score in zip(bars1, country_summary['avg_health_score']):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height + 2,
             f'{score:.1f}',
             ha='center', va='bottom', fontsize=12, fontweight='bold')

# Panel 2: Rating Distribution Stacked Bar
ax2 = axes[0, 1]
colors_rating = {
    'EXCELLENT': '#63E6BE',
    'GOOD': '#6BCB77',
    'FAIR': '#FFD93D',
    'POOR': '#FFA07A',
    'CRITICAL': '#FF9FF3'
}

rating_dist_sorted = rating_dist.reindex(country_summary['country'])
bottom = np.zeros(len(rating_dist_sorted))

for rating in rating_order:
    if rating in rating_dist_sorted.columns:
        ax2.bar(rating_dist_sorted.index, rating_dist_sorted[rating], 
                bottom=bottom, label=rating, color=colors_rating[rating], 
                edgecolor='white', linewidth=1.5)
        bottom += rating_dist_sorted[rating]

ax2.set_ylabel('Number of Region-Days', fontsize=12, fontweight='bold')
ax2.set_title('Rating Distribution by Country', fontsize=13, fontweight='bold')
ax2.legend(loc='upper right', fontsize=9)
ax2.grid(axis='y', alpha=0.3)

# Panel 3: Total Incidents by Country
ax3 = axes[1, 0]
bars3 = ax3.bar(country_summary['country'], country_summary['total_incidents'], 
                color='#FFB86C', edgecolor='white', linewidth=2)
ax3.set_ylabel('Total Incidents', fontsize=12, fontweight='bold')
ax3.set_title('Total Incidents by Country', fontsize=13, fontweight='bold')
ax3.grid(axis='y', alpha=0.3)

# Add value labels
for bar, count in zip(bars3, country_summary['total_incidents']):
    height = bar.get_height()
    ax3.text(bar.get_x() + bar.get_width()/2., height + 1,
             f'{int(count)}',
             ha='center', va='bottom', fontsize=12, fontweight='bold')

# Panel 4: Total Customers Affected
ax4 = axes[1, 1]
bars4 = ax4.bar(country_summary['country'], country_summary['total_customers'], 
                color='#B197FC', edgecolor='white', linewidth=2)
ax4.set_ylabel('Total Customers Affected', fontsize=12, fontweight='bold')
ax4.set_title('Customer Impact by Country', fontsize=13, fontweight='bold')
ax4.grid(axis='y', alpha=0.3)

# Add value labels
for bar, customers in zip(bars4, country_summary['total_customers']):
    height = bar.get_height()
    ax4.text(bar.get_x() + bar.get_width()/2., height + 2000,
             f'{int(customers/1000):.0f}K',
             ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.suptitle('Country Operational Performance Summary', fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("COUNTRY OPERATIONAL RATINGS SUMMARY")
print("="*80)

for idx, row in country_summary.iterrows():
    country = row['country']
    print(f"\n{country}:")
    print(f"  Average Health Score: {row['avg_health_score']:.1f}")
    
    # Determine overall rating
    if row['avg_health_score'] >= 60:
        rating = "GOOD"
    elif row['avg_health_score'] >= 40:
        rating = "FAIR"
    elif row['avg_health_score'] >= 20:
        rating = "POOR"
    else:
        rating = "CRITICAL"
    print(f"  Overall Rating: {rating}")
    
    print(f"  Total Incidents: {int(row['total_incidents'])}")
    print(f"  Customers Affected: {int(row['total_customers']):,}")
    print(f"  Days Analyzed: {int(row['num_region_days'])}")
    
    # Show rating distribution
    if country in rating_dist.index:
        print(f"  Rating Distribution:")
        for rating in rating_order:
            if rating in rating_dist.columns:
                count = rating_dist.loc[country, rating]
                if count > 0:
                    pct = (count / row['num_region_days']) * 100
                    print(f"    - {rating}: {int(count)} days ({pct:.0f}%)")

print("\n" + "="*80)
print("🎯 KEY FINDINGS:")
print("="*80)

if not country_summary.empty:
    # Find best and worst
    best_country = country_summary.iloc[0]
    worst_country = country_summary.iloc[-1]
    
    print(f"\n🏆 Best Performer: {best_country['country']}")
    print(f"   - Avg Score: {best_country['avg_health_score']:.1f}")
    print(f"   - Total Incidents: {int(best_country['total_incidents'])}")
    
    print(f"\n🚨 Worst Performer: {worst_country['country']}")
    print(f"   - Avg Score: {worst_country['avg_health_score']:.1f}")
    print(f"   - Total Incidents: {int(worst_country['total_incidents'])}")
    print(f"   - Customers Affected: {int(worst_country['total_customers']):,}")
    
    # Count GOOD and EXCELLENT days across all countries
    total_days = daily_ratings.shape[0]
    good_excellent_days = len(daily_ratings[daily_ratings['operational_condition'].isin(['GOOD', 'EXCELLENT'])])
    poor_critical_days = len(daily_ratings[daily_ratings['operational_condition'].isin(['POOR', 'CRITICAL'])])
    
    print(f"\n🚨 OVERALL GRID HEALTH:")
    print(f"   - Total region-days: {total_days}")
    print(f"   - GOOD/EXCELLENT days: {good_excellent_days} ({good_excellent_days/total_days*100:.1f}%)")
    print(f"   - POOR/CRITICAL days: {poor_critical_days} ({poor_critical_days/total_days*100:.1f}%)")
    if good_excellent_days < 5:
        print(f"   ⚠️ WARNING: Grid operates in stressed state {poor_critical_days/total_days*100:.0f}% of the time!")

print("="*80)

## ⏰ Question 6: When Do Grid Failures Happen?

**Temporal Analysis:**
- Hour of day patterns (peak demand correlation?)
- Weekend vs weekday response
- Trends over time (improving or degrading?)
- Day clustering (cascade effects?)

**What we'll discover:** Are failures predictable based on time patterns?

In [0]:
# Analyze temporal patterns
temporal_analysis = df_operations.select(
    "event_id", "timestamp", "region_name", "duration_minutes", 
    "affected_customers", "severity", "hour", "day_of_week", "is_weekend", "event_day"
).toPandas()

temporal_analysis['timestamp'] = pd.to_datetime(temporal_analysis['timestamp'])
temporal_analysis['event_day'] = pd.to_datetime(temporal_analysis['event_day'])

# Create 3-panel visualization
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# Panel 1: Hour of Day Distribution
ax1 = axes[0, 0]
hour_counts = temporal_analysis.groupby('hour').size()
hours = range(24)
counts = [hour_counts.get(h, 0) for h in hours]

# Color code by time of day - CHEERFUL COLORS
colors_hour = ['#B197FC' if h < 6 or h >= 22 else '#63E6BE' if h < 12 else '#FFD93D' if h < 18 else '#FF6B9D' 
               for h in hours]

ax1.bar(hours, counts, color=colors_hour, edgecolor='white', linewidth=2)
ax1.set_xlabel('Hour of Day (24h format)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Number of Incidents', fontsize=12, fontweight='bold')
ax1.set_title('Incident Distribution by Hour of Day', fontsize=14, fontweight='bold')
ax1.set_xticks(range(0, 24, 2))
ax1.grid(axis='y', alpha=0.3)

# Add time period labels
ax1.text(0.05, 0.95, 'Night (22-6): Purple\nMorning (6-12): Mint\nAfternoon (12-18): Sunshine\nEvening (18-22): Pink', 
         transform=ax1.transAxes, fontsize=9, verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='#B197FC', linewidth=2))

# Panel 2: Day of Week Distribution
ax2 = axes[0, 1]
day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
day_counts = temporal_analysis.groupby('day_of_week').size().reindex(range(1, 8), fill_value=0)
colors_day = ['#4ECDC4' if d <= 5 else '#FF9FF3' for d in range(1, 8)]

ax2.bar(range(7), day_counts.values, color=colors_day, edgecolor='white', linewidth=2)
ax2.set_xlabel('Day of Week', fontsize=12, fontweight='bold')
ax2.set_ylabel('Number of Incidents', fontsize=12, fontweight='bold')
ax2.set_title('Incident Distribution by Day of Week', fontsize=14, fontweight='bold')
ax2.set_xticks(range(7))
ax2.set_xticklabels(day_names)
ax2.grid(axis='y', alpha=0.3)

# Add value labels
for i, count in enumerate(day_counts.values):
    if count > 0:
        ax2.text(i, count + 0.1, str(int(count)), ha='center', va='bottom', fontsize=11, fontweight='bold')

# Weekend comparison
weekday_avg = temporal_analysis[temporal_analysis['is_weekend'] == 0].shape[0] / 5
weekend_avg = temporal_analysis[temporal_analysis['is_weekend'] == 1].shape[0] / 2
ax2.text(0.98, 0.95, f'Weekday avg: {weekday_avg:.1f}/day\nWeekend avg: {weekend_avg:.1f}/day', 
         transform=ax2.transAxes, fontsize=10, verticalalignment='top', horizontalalignment='right',
         bbox=dict(boxstyle='round', facecolor='#FFF9E6', alpha=0.9, edgecolor='#FFD93D', linewidth=2))

# Panel 3: Timeline - Incidents per Day
ax3 = axes[1, 0]
daily_incidents = temporal_analysis.groupby('event_day').size().reset_index(name='count')
all_days = pd.date_range(start=temporal_analysis['event_day'].min(), 
                         end=temporal_analysis['event_day'].max(), freq='D')
daily_full = pd.DataFrame({'event_day': all_days})
daily_full = daily_full.merge(daily_incidents, on='event_day', how='left').fillna(0)

colors_timeline = ['#FF6B9D' if c >= 2 else '#FFD93D' if c == 1 else '#B4E7CE' for c in daily_full['count']]
ax3.bar(daily_full['event_day'], daily_full['count'], color=colors_timeline, edgecolor='white', linewidth=1.5)
ax3.set_xlabel('Date (January 2024)', fontsize=12, fontweight='bold')
ax3.set_ylabel('Incidents per Day', fontsize=12, fontweight='bold')
ax3.set_title('Daily Incident Trend: Are Failures Clustered?', fontsize=14, fontweight='bold')
ax3.grid(axis='y', alpha=0.3)
ax3.tick_params(axis='x', rotation=45)

# Highlight multi-incident days
multi_days = daily_full[daily_full['count'] >= 2]['event_day']
if len(multi_days) > 0:
    ax3.text(0.02, 0.95, f'{len(multi_days)} days with 2+ incidents\n(potential cascade effects)', 
             transform=ax3.transAxes, fontsize=10, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='#FFE5F1', alpha=0.9, edgecolor='#FF6B9D', linewidth=2))

# Panel 4: Weekend vs Weekday Response Time
ax4 = axes[1, 1]
weekday_data = temporal_analysis[temporal_analysis['is_weekend'] == 0]
weekend_data = temporal_analysis[temporal_analysis['is_weekend'] == 1]

categories = ['Weekday', 'Weekend']
avg_duration = [
    weekday_data['duration_minutes'].mean() if len(weekday_data) > 0 else 0,
    weekend_data['duration_minutes'].mean() if len(weekend_data) > 0 else 0
]
avg_customers = [
    weekday_data['affected_customers'].mean() if len(weekday_data) > 0 else 0,
    weekend_data['affected_customers'].mean() if len(weekend_data) > 0 else 0
]

x = np.arange(len(categories))
width = 0.35

ax4_twin = ax4.twinx()
bars1 = ax4.bar(x - width/2, avg_duration, width, label='Avg Duration (min)', color='#FFA07A', edgecolor='white', linewidth=2)
bars2 = ax4_twin.bar(x + width/2, avg_customers, width, label='Avg Customers', color='#4ECDC4', edgecolor='white', linewidth=2)

ax4.set_ylabel('Average Duration (minutes)', fontsize=12, fontweight='bold', color='#FFA07A')
ax4_twin.set_ylabel('Average Customers Affected', fontsize=12, fontweight='bold', color='#4ECDC4')
ax4.set_xlabel('Day Type', fontsize=12, fontweight='bold')
ax4.set_title('Weekend vs Weekday Incident Severity', fontsize=14, fontweight='bold')
ax4.set_xticks(x)
ax4.set_xticklabels(categories)
ax4.tick_params(axis='y', labelcolor='#FFA07A')
ax4_twin.tick_params(axis='y', labelcolor='#4ECDC4')
ax4.grid(axis='y', alpha=0.3)

# Add value labels
for i, (bar, val) in enumerate(zip(bars1, avg_duration)):
    ax4.text(bar.get_x() + bar.get_width()/2., val + 5, f'{val:.0f}',
             ha='center', va='bottom', fontsize=11, fontweight='bold', color='#FFA07A')
for i, (bar, val) in enumerate(zip(bars2, avg_customers)):
    ax4_twin.text(bar.get_x() + bar.get_width()/2., val + 100, f'{val:.0f}',
                  ha='center', va='bottom', fontsize=11, fontweight='bold', color='#4ECDC4')

# Combined legend
lines1, labels1 = ax4.get_legend_handles_labels()
lines2, labels2 = ax4_twin.get_legend_handles_labels()
ax4.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=10)

plt.suptitle('Temporal Patterns: When Do Grid Failures Happen?', fontsize=17, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("TEMPORAL ANALYSIS")
print("="*80)

print("\n1. HOUR OF DAY PATTERNS:")
peak_hour = hour_counts.idxmax() if len(hour_counts) > 0 else None
if peak_hour is not None:
    print(f"   Peak incident hour: {peak_hour}:00 ({hour_counts[peak_hour]} incidents)")
# Use list() to materialize generator before passing to builtin sum
night_incidents = int(np.sum([hour_counts.get(h, 0) for h in list(range(22, 24)) + list(range(0, 6))]))
day_incidents = int(np.sum([hour_counts.get(h, 0) for h in range(6, 22)]))
print(f"   Night incidents (22:00-6:00): {night_incidents}")
print(f"   Day incidents (6:00-22:00): {day_incidents}")

print("\n2. WEEKEND VS WEEKDAY:")
print(f"   Weekday incidents: {len(weekday_data)} ({len(weekday_data)/len(temporal_analysis)*100:.0f}%)")
print(f"   Weekend incidents: {len(weekend_data)} ({len(weekend_data)/len(temporal_analysis)*100:.0f}%)")
print(f"   Weekday avg duration: {avg_duration[0]:.0f} min")
print(f"   Weekend avg duration: {avg_duration[1]:.0f} min")
if avg_duration[1] > avg_duration[0]:
    print(f"   → Weekend incidents take {avg_duration[1]/avg_duration[0]:.1f}× longer to resolve!")

print("\n3. CLUSTERING ANALYSIS:")
print(f"   Total days in period: {len(daily_full)}")
print(f"   Days with incidents: {len(daily_full[daily_full['count'] > 0])}")
print(f"   Days with 2+ incidents: {len(multi_days)} (potential cascades)")
print(f"   Max incidents in one day: {int(daily_full['count'].max())}")

print("\n4. TREND OVER TIME:")
first_week = temporal_analysis[temporal_analysis['event_day'] <= temporal_analysis['event_day'].min() + pd.Timedelta(days=7)]
last_week = temporal_analysis[temporal_analysis['event_day'] >= temporal_analysis['event_day'].max() - pd.Timedelta(days=7)]
print(f"   First week incidents: {len(first_week)}")
print(f"   Last week incidents: {len(last_week)}")
if len(last_week) > len(first_week):
    print(f"   → Performance DEGRADING over time ({len(last_week) - len(first_week)} more incidents)")
elif len(last_week) < len(first_week):
    print(f"   → Performance IMPROVING over time ({len(first_week) - len(last_week)} fewer incidents)")
else:
    print(f"   → Performance STABLE over time")

print("\n" + "="*80)
print("🎯 KEY INSIGHTS:")
print("="*80)
if peak_hour is not None:
    print(f"\n• Incidents cluster around {peak_hour}:00 (operational vulnerability)")
if len(weekend_data) > 0 and avg_duration[1] > avg_duration[0] * 1.2:
    print(f"• Weekend response is {avg_duration[1]/avg_duration[0]:.1f}× slower (staffing issue?)")
if len(multi_days) > 0:
    print(f"• {len(multi_days)} days with multiple incidents suggest cascade effects")
print("="*80)

## 👥 Question 7: The Customer Impact Story

**Quantifying the Human Cost:**
- Top 5 most devastating incidents (customer-hours lost)
- Which regions suffer most?
- Total economic impact
- Single-event vs cumulative damage

**What we'll discover:** The real-world consequences of grid failures

In [0]:
# Calculate customer impact metrics
impact_analysis = df_operations.select(
    "event_id", "timestamp", "region_name", "substation_id",
    "duration_minutes", "affected_customers", "severity"
).toPandas()

# Calculate customer-hours lost (duration × customers)
impact_analysis['customer_hours_lost'] = (impact_analysis['duration_minutes'] / 60) * impact_analysis['affected_customers']
impact_analysis['timestamp'] = pd.to_datetime(impact_analysis['timestamp'])

# Sort by impact
impact_analysis = impact_analysis.sort_values('customer_hours_lost', ascending=False)

# Create 2×2 visualization with CHEERFUL COLORS
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# Panel 1: Top 10 Most Devastating Incidents (Bubble Chart)
ax1 = axes[0, 0]
top10_impact = impact_analysis.head(10)

# Define cheerful colors for regions
region_colors = {
    'Copenhagen': '#FF6B9D',
    'Helsinki': '#FFB86C', 
    'Turku': '#FFD93D',
    'Gothenburg': '#4ECDC4',
    'Tampere': '#B197FC',
    'Malmo': '#6BCB77'
}

for idx, row in top10_impact.iterrows():
    color = region_colors.get(row['region_name'], '#B4E7CE')
    # Bubble size represents customer-hours
    size = row['customer_hours_lost'] / 100
    ax1.scatter(row['duration_minutes'], row['affected_customers'], 
               s=size, color=color, alpha=0.7, edgecolors='white', linewidth=2)
    # Label with event ID
    ax1.text(row['duration_minutes'] + 10, row['affected_customers'] + 100, 
            f"{row['substation_id']}\n{row['customer_hours_lost']:.0f}h",
            fontsize=8, fontweight='bold')

ax1.set_xlabel('Outage Duration (minutes)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Customers Affected', fontsize=12, fontweight='bold')
ax1.set_title('Top 10 Most Devastating Incidents\n(Bubble size = Customer-Hours Lost)', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)

# Add legend for regions
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=color, edgecolor='white', label=region) 
                   for region, color in region_colors.items()]
ax1.legend(handles=legend_elements, loc='upper right', fontsize=9, title='Region')

# Panel 2: Customer-Hours Lost by Region (Total Impact)
ax2 = axes[0, 1]
regional_impact = impact_analysis.groupby('region_name').agg({
    'customer_hours_lost': 'sum',
    'affected_customers': 'sum',
    'duration_minutes': 'mean'
}).sort_values('customer_hours_lost', ascending=True)

y_pos = np.arange(len(regional_impact))
colors = [region_colors.get(region, '#B4E7CE') for region in regional_impact.index]

bars = ax2.barh(y_pos, regional_impact['customer_hours_lost'], color=colors, edgecolor='white', linewidth=2)
ax2.set_yticks(y_pos)
ax2.set_yticklabels(regional_impact.index, fontsize=11)
ax2.set_xlabel('Total Customer-Hours Lost', fontsize=12, fontweight='bold')
ax2.set_title('Regional Suffering: Who Pays the Price?', fontsize=14, fontweight='bold')
ax2.grid(axis='x', alpha=0.3)

# Add value labels
for i, (bar, val) in enumerate(zip(bars, regional_impact['customer_hours_lost'])):
    ax2.text(val + 500, bar.get_y() + bar.get_height()/2, f'{val:.0f}h',
             va='center', fontsize=11, fontweight='bold')

# Panel 3: Impact Distribution (Severity Breakdown)
ax3 = axes[1, 0]
severity_impact = impact_analysis.groupby('severity').agg({
    'customer_hours_lost': 'sum',
    'event_id': 'count'
}).rename(columns={'event_id': 'incident_count'})
severity_impact = severity_impact.reindex(['critical', 'medium', 'low'], fill_value=0)

colors_severity = ['#FF9FF3', '#FFD93D', '#B4E7CE']
x = np.arange(len(severity_impact))
width = 0.35

ax3_twin = ax3.twinx()
bars1 = ax3.bar(x - width/2, severity_impact['incident_count'], width, 
               label='Incident Count', color='#4ECDC4', edgecolor='white', linewidth=2)
bars2 = ax3_twin.bar(x + width/2, severity_impact['customer_hours_lost'], width,
                    label='Customer-Hours Lost', color=colors_severity, edgecolor='white', linewidth=2)

ax3.set_ylabel('Number of Incidents', fontsize=12, fontweight='bold', color='#4ECDC4')
ax3_twin.set_ylabel('Customer-Hours Lost', fontsize=12, fontweight='bold', color='#FF9FF3')
ax3.set_xlabel('Severity', fontsize=12, fontweight='bold')
ax3.set_title('Impact by Severity: Critical Incidents Dominate', fontsize=14, fontweight='bold')
ax3.set_xticks(x)
ax3.set_xticklabels(['Critical', 'Medium', 'Low'])
ax3.tick_params(axis='y', labelcolor='#4ECDC4')
ax3_twin.tick_params(axis='y', labelcolor='#FF9FF3')
ax3.grid(axis='y', alpha=0.3)

# Add value labels
for bar, val in zip(bars1, severity_impact['incident_count']):
    if val > 0:
        ax3.text(bar.get_x() + bar.get_width()/2., val + 0.2, f'{int(val)}',
                ha='center', va='bottom', fontsize=11, fontweight='bold', color='#4ECDC4')
for bar, val in zip(bars2, severity_impact['customer_hours_lost']):
    if val > 0:
        ax3_twin.text(bar.get_x() + bar.get_width()/2., val + 500, f'{val:.0f}h',
                     ha='center', va='bottom', fontsize=11, fontweight='bold', color='#FF9FF3')

# Combined legend
lines1, labels1 = ax3.get_legend_handles_labels()
lines2, labels2 = ax3_twin.get_legend_handles_labels()
ax3.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=10)

# Panel 4: Cumulative Impact Timeline
ax4 = axes[1, 1]
impact_timeline = impact_analysis.sort_values('timestamp')
impact_timeline['cumulative_customer_hours'] = impact_timeline['customer_hours_lost'].cumsum()

ax4.plot(impact_timeline['timestamp'], impact_timeline['cumulative_customer_hours'], 
        color='#FF6B9D', linewidth=3, marker='o', markersize=6, markeredgecolor='white', markeredgewidth=1.5)
ax4.fill_between(impact_timeline['timestamp'], impact_timeline['cumulative_customer_hours'], 
                alpha=0.3, color='#FFB5D5')
ax4.set_xlabel('Date (January 2024)', fontsize=12, fontweight='bold')
ax4.set_ylabel('Cumulative Customer-Hours Lost', fontsize=12, fontweight='bold')
ax4.set_title('The Accumulating Human Cost', fontsize=14, fontweight='bold')
ax4.grid(True, alpha=0.3)
ax4.tick_params(axis='x', rotation=45)

# Highlight major jumps
impact_timeline['daily_impact'] = impact_timeline['customer_hours_lost']
max_jump = impact_timeline.loc[impact_timeline['daily_impact'].idxmax()]
ax4.annotate(f"Biggest impact:\n{max_jump['substation_id']}\n{max_jump['customer_hours_lost']:.0f}h",
            xy=(max_jump['timestamp'], max_jump['cumulative_customer_hours']),
            xytext=(max_jump['timestamp'] - pd.Timedelta(days=2), max_jump['cumulative_customer_hours'] + 5000),
            fontsize=10, fontweight='bold', color='#FF6B9D',
            bbox=dict(boxstyle='round', facecolor='#FFE5F1', edgecolor='#FF6B9D', linewidth=2),
            arrowprops=dict(arrowstyle='->', lw=2, color='#FF6B9D'))

plt.suptitle('The Customer Impact Story: Quantifying the Human Cost', fontsize=17, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("CUSTOMER IMPACT ANALYSIS")
print("="*80)

print("\n1. TOP 5 MOST DEVASTATING INCIDENTS:")
top5 = impact_analysis.head(5)
for idx, (_, row) in enumerate(top5.iterrows(), 1):
    print(f"\n   #{idx}: {row['substation_id']} ({row['region_name']})")
    print(f"      - Customer-Hours Lost: {row['customer_hours_lost']:.0f}h")
    print(f"      - Duration: {row['duration_minutes']:.0f} min")
    print(f"      - Customers: {int(row['affected_customers']):,}")
    print(f"      - Severity: {row['severity']}")

print("\n2. REGIONAL SUFFERING:")
for region, data in regional_impact.iterrows():
    total_customers = int(data['affected_customers'])
    total_hours = data['customer_hours_lost']
    avg_duration = data['duration_minutes']
    print(f"\n   {region}:")
    print(f"      - Total customer-hours lost: {total_hours:.0f}h")
    print(f"      - Total customers affected: {total_customers:,}")
    print(f"      - Avg outage duration: {avg_duration:.0f} min")

worst_region = regional_impact.idxmax()['customer_hours_lost']
print(f"\n   → {worst_region} suffered the most: {regional_impact.loc[worst_region, 'customer_hours_lost']:.0f}h lost")

print("\n3. SEVERITY BREAKDOWN:")
total_customer_hours = impact_analysis['customer_hours_lost'].sum()
for severity, data in severity_impact.iterrows():
    pct = (data['customer_hours_lost'] / total_customer_hours) * 100 if total_customer_hours > 0 else 0
    print(f"   {severity.capitalize()}: {int(data['incident_count'])} incidents, {data['customer_hours_lost']:.0f}h ({pct:.0f}%)")

print("\n4. TOTAL ECONOMIC IMPACT:")
print(f"   Total customer-hours lost: {total_customer_hours:.0f}h")
print(f"   Total customers affected: {impact_analysis['affected_customers'].sum():,}")
print(f"   Average customer-hours per incident: {total_customer_hours / len(impact_analysis):.0f}h")

# Estimate economic cost (assuming $10/customer-hour for lost productivity/comfort)
economic_cost = total_customer_hours * 10
print(f"   Estimated economic cost (@$10/customer-hour): ${economic_cost:,.0f}")

print("\n" + "="*80)
print("🎯 KEY INSIGHTS:")
print("="*80)
print(f"\n• Top incident: {top5.iloc[0]['substation_id']} caused {top5.iloc[0]['customer_hours_lost']:.0f}h of disruption")
print(f"• {worst_region} region bears the greatest burden ({regional_impact.loc[worst_region, 'customer_hours_lost']:.0f}h)")
critical_pct = (severity_impact.loc['critical', 'customer_hours_lost'] / total_customer_hours) * 100 if 'critical' in severity_impact.index else 0
print(f"• Critical incidents account for {critical_pct:.0f}% of total customer impact")
print(f"• Economic cost: ${economic_cost:,.0f} in lost productivity")
print("="*80)

## 🏭 Question 8: Asset Portfolio Health

**Understanding the Infrastructure:**
- Age distribution across the portfolio
- Risk concentration (Pareto principle)
- Investment priorities
- Modern vs aging infrastructure performance

**What we'll discover:** Where to focus modernization efforts

In [0]:
# Load asset reference data and join with operations
asset_ref = spark.table("vattenfall_dev.refined.silver_asset_reference").select(
    "substation_id", "asset_age_years"
).toPandas()
operations_pd = df_operations.select(
    "event_id", "substation_id", "region_name", "duration_minutes", "affected_customers"
).toPandas()

# Merge to get asset details
asset_analysis = operations_pd.merge(
    asset_ref,
    on='substation_id',
    how='left'
)

# Calculate asset-level metrics
asset_metrics = asset_analysis.groupby('substation_id').agg({
    'event_id': 'count',
    'duration_minutes': 'mean',
    'affected_customers': 'sum',
    'asset_age_years': 'first',
    'region_name': 'first'
}).rename(columns={'event_id': 'incident_count'}).reset_index()

# Calculate performance score
asset_metrics['performance_score'] = (
    asset_metrics['incident_count'] * 10 +
    asset_metrics['duration_minutes'] * 0.5 +
    asset_metrics['affected_customers'] / 100 +
    asset_metrics['asset_age_years'] * 0.5
)

# Create 2x2 visualization with CHEERFUL COLORS
fig = plt.figure(figsize=(18, 12))
gs = fig.add_gridspec(2, 2, hspace=0.3, wspace=0.3)

# Panel 1: Age Distribution (Histogram with Risk Zones)
ax1 = fig.add_subplot(gs[0, 0])

age_bins = [0, 5, 10, 15, 20, 25, 35]
age_labels = ['0-5yr\n(New)', '5-10yr\n(Good)', '10-15yr\n(Aging)', '15-20yr\n(High Risk)', '20-25yr\n(Critical)', '25+yr\n(End of Life)']
age_colors = ['#63E6BE', '#6BCB77', '#FFD93D', '#FFB86C', '#FFA07A', '#FF9FF3']

asset_metrics['age_category'] = pd.cut(asset_metrics['asset_age_years'], bins=age_bins, labels=age_labels, include_lowest=True)
age_dist = asset_metrics['age_category'].value_counts().reindex(age_labels, fill_value=0)

bars = ax1.bar(range(len(age_dist)), age_dist.values, color=age_colors, edgecolor='white', linewidth=2)
ax1.set_xticks(range(len(age_dist)))
ax1.set_xticklabels(age_labels, fontsize=10)
ax1.set_ylabel('Number of Assets', fontsize=12, fontweight='bold')
ax1.set_title('Asset Age Distribution: Risk Profile', fontsize=14, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# Add value labels
for bar, val in zip(bars, age_dist.values):
    if val > 0:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height + 0.2,
                f'{int(val)}\n({val/len(asset_metrics)*100:.0f}%)',
                ha='center', va='bottom', fontsize=10, fontweight='bold')

# Add risk zone shading
ax1.axvspan(-0.5, 1.5, alpha=0.1, color='#63E6BE', label='Low Risk Zone')
ax1.axvspan(1.5, 2.5, alpha=0.1, color='#FFD93D', label='Moderate Risk')
ax1.axvspan(2.5, 5.5, alpha=0.1, color='#FF9FF3', label='High Risk Zone')
ax1.legend(loc='upper right', fontsize=9)

# Panel 2: Risk Concentration (Pareto Chart)
ax2 = fig.add_subplot(gs[0, 1])
ax2_twin = ax2.twinx()

# Sort by performance score
asset_sorted = asset_metrics.sort_values('performance_score', ascending=False).reset_index(drop=True)
asset_sorted['cumulative_pct'] = (asset_sorted['performance_score'].cumsum() / asset_sorted['performance_score'].sum()) * 100

# Show top 15 assets (use np.minimum to avoid PySpark conflict)
top_n = int(np.minimum(15, len(asset_sorted)))
top_assets = asset_sorted.head(top_n)

# Bar chart for individual scores
bars = ax2.bar(range(len(top_assets)), top_assets['performance_score'], 
              color='#FF9FF3', alpha=0.7, edgecolor='white', linewidth=2, label='Performance Score')

# Line chart for cumulative percentage
line = ax2_twin.plot(range(len(top_assets)), top_assets['cumulative_pct'], 
                    color='#4ECDC4', linewidth=3, marker='o', markersize=8, 
                    markeredgecolor='white', markeredgewidth=2, label='Cumulative %')
ax2_twin.axhline(y=80, color='#6BCB77', linestyle='--', linewidth=2, label='80% Threshold')

ax2.set_xlabel('Asset Rank (Worst to Best)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Performance Score', fontsize=12, fontweight='bold', color='#FF9FF3')
ax2_twin.set_ylabel('Cumulative % of Total Risk', fontsize=12, fontweight='bold', color='#4ECDC4')
ax2.set_title('Risk Concentration: Pareto Analysis', fontsize=14, fontweight='bold')
ax2.set_xticks(range(0, len(top_assets), 2))
ax2.set_xticklabels(range(1, len(top_assets)+1, 2))
ax2.tick_params(axis='y', labelcolor='#FF9FF3')
ax2_twin.tick_params(axis='y', labelcolor='#4ECDC4')
ax2.grid(axis='y', alpha=0.3)

# Find 80/20 point
eighty_pct_idx = (asset_sorted['cumulative_pct'] >= 80).idxmax()
pct_assets_for_80 = (eighty_pct_idx / len(asset_sorted)) * 100
if eighty_pct_idx < len(top_assets):
    ax2_twin.axvline(x=eighty_pct_idx, color='#6BCB77', linestyle='--', linewidth=2, alpha=0.5)
    ax2_twin.text(eighty_pct_idx + 1, 50, f'{pct_assets_for_80:.0f}% of assets\ncause 80% of risk',
                 fontsize=10, fontweight='bold', color='#6BCB77',
                 bbox=dict(boxstyle='round', facecolor='#E8F8F5', edgecolor='#6BCB77', linewidth=2))

# Combined legend
lines1, labels1 = ax2.get_legend_handles_labels()
lines2, labels2 = ax2_twin.get_legend_handles_labels()
ax2.legend(lines1 + lines2, labels1 + labels2, loc='center right', fontsize=9)

# Panel 3: Age vs Performance Scatter (All Assets)
ax3 = fig.add_subplot(gs[1, 0])

# Color by incident count
scatter = ax3.scatter(asset_metrics['asset_age_years'], asset_metrics['performance_score'],
                     s=asset_metrics['affected_customers']/50, 
                     c=asset_metrics['incident_count'],
                     cmap='RdYlGn_r', alpha=0.7, edgecolors='white', linewidth=1.5)

ax3.set_xlabel('Asset Age (years)', fontsize=12, fontweight='bold')
ax3.set_ylabel('Performance Score', fontsize=12, fontweight='bold')
ax3.set_title('Age vs Performance: Not Always Correlated', fontsize=14, fontweight='bold')
ax3.grid(True, alpha=0.3)

# Add colorbar for incident count
cbar = plt.colorbar(scatter, ax=ax3)
cbar.set_label('Incident Count', fontsize=10, fontweight='bold')

# Add size legend
for size, label in [(500, '2,500'), (1500, '7,500'), (2500, '12,500')]:
    ax3.scatter([], [], s=size/50, c='gray', alpha=0.5, edgecolors='white', linewidth=1.5, label=f'{label} customers')
ax3.legend(frameon=True, labelspacing=2, title='Affected Customers', loc='upper left', fontsize=9)

# Highlight worst performers
worst_3 = asset_metrics.nlargest(3, 'performance_score')
for _, asset in worst_3.iterrows():
    ax3.annotate(asset['substation_id'],
                xy=(asset['asset_age_years'], asset['performance_score']),
                xytext=(asset['asset_age_years'] + 2, asset['performance_score'] + 10),
                fontsize=9, fontweight='bold', color='#FF6B9D',
                bbox=dict(boxstyle='round', facecolor='#FFE5F1', edgecolor='#FF6B9D', linewidth=1.5),
                arrowprops=dict(arrowstyle='->', lw=1.5, color='#FF6B9D'))

# Panel 4: Investment Priority Matrix
ax4 = fig.add_subplot(gs[1, 1])

# Categorize assets by age and performance
age_threshold = asset_metrics['asset_age_years'].median()
perf_threshold = asset_metrics['performance_score'].median()

asset_metrics['category'] = 'Monitor'
asset_metrics.loc[(asset_metrics['asset_age_years'] >= age_threshold) & 
                 (asset_metrics['performance_score'] >= perf_threshold), 'category'] = 'Replace Now'
asset_metrics.loc[(asset_metrics['asset_age_years'] < age_threshold) & 
                 (asset_metrics['performance_score'] >= perf_threshold), 'category'] = 'Investigate'
asset_metrics.loc[(asset_metrics['asset_age_years'] >= age_threshold) & 
                 (asset_metrics['performance_score'] < perf_threshold), 'category'] = 'Upgrade Soon'

category_colors = {
    'Replace Now': '#FF6B9D',
    'Investigate': '#FFD93D', 
    'Upgrade Soon': '#4ECDC4',
    'Monitor': '#6BCB77'
}

for category, color in category_colors.items():
    subset = asset_metrics[asset_metrics['category'] == category]
    ax4.scatter(subset['asset_age_years'], subset['performance_score'],
               s=subset['affected_customers']/30, color=color, alpha=0.7,
               edgecolors='white', linewidth=1.5, label=category)

# Add quadrant lines
ax4.axhline(y=perf_threshold, color='gray', linestyle='--', linewidth=2, alpha=0.5)
ax4.axvline(x=age_threshold, color='gray', linestyle='--', linewidth=2, alpha=0.5)

# Add quadrant labels
ax4.text(5, asset_metrics['performance_score'].max() * 0.95, 'Young but\nProblematic',
        fontsize=11, fontweight='bold', ha='center', color='#FFD93D',
        bbox=dict(boxstyle='round', facecolor='#FFF9E6', edgecolor='#FFD93D', linewidth=2))
ax4.text(age_threshold * 1.5, asset_metrics['performance_score'].max() * 0.95, 'Old and\nProblematic',
        fontsize=11, fontweight='bold', ha='center', color='#FF6B9D',
        bbox=dict(boxstyle='round', facecolor='#FFE5F1', edgecolor='#FF6B9D', linewidth=2))
ax4.text(5, perf_threshold * 0.4, 'Young and\nReliable',
        fontsize=11, fontweight='bold', ha='center', color='#6BCB77',
        bbox=dict(boxstyle='round', facecolor='#E8F8F5', edgecolor='#6BCB77', linewidth=2))
ax4.text(age_threshold * 1.5, perf_threshold * 0.4, 'Old but\nStable',
        fontsize=11, fontweight='bold', ha='center', color='#4ECDC4',
        bbox=dict(boxstyle='round', facecolor='#E3F8FF', edgecolor='#4ECDC4', linewidth=2))

ax4.set_xlabel('Asset Age (years)', fontsize=12, fontweight='bold')
ax4.set_ylabel('Performance Score', fontsize=12, fontweight='bold')
ax4.set_title('Investment Priority Matrix', fontsize=14, fontweight='bold')
ax4.legend(loc='lower right', fontsize=10, title='Action Required')
ax4.grid(True, alpha=0.3)

plt.suptitle('Asset Portfolio Health: Strategic Investment Priorities', fontsize=17, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("ASSET PORTFOLIO HEALTH ANALYSIS")
print("="*80)

print("\n1. AGE DISTRIBUTION:")
for age_cat, count in age_dist.items():
    pct = (count / len(asset_metrics)) * 100
    print(f"   {age_cat}: {count} assets ({pct:.0f}%)")

high_risk_count = age_dist[age_dist.index.str.contains('High Risk|Critical|End of Life')].sum()
high_risk_pct = (high_risk_count / len(asset_metrics)) * 100
print(f"\n   → {high_risk_count} assets ({high_risk_pct:.0f}%) in high-risk age categories")

print("\n2. RISK CONCENTRATION (PARETO PRINCIPLE):")
print(f"   Total assets: {len(asset_metrics)}")
print(f"   Assets causing 80% of risk: {eighty_pct_idx + 1} ({pct_assets_for_80:.0f}%)")
print(f"\n   Top 5 risk contributors:")
for idx, (_, asset) in enumerate(asset_sorted.head(5).iterrows(), 1):
    print(f"      #{idx}: {asset['substation_id']} (Score: {asset['performance_score']:.1f}, Age: {asset['asset_age_years']:.0f}yr)")

print("\n3. INVESTMENT PRIORITIES:")
for category in ['Replace Now', 'Investigate', 'Upgrade Soon', 'Monitor']:
    count = (asset_metrics['category'] == category).sum()
    pct = (count / len(asset_metrics)) * 100
    avg_age = asset_metrics[asset_metrics['category'] == category]['asset_age_years'].mean()
    print(f"\n   {category}: {count} assets ({pct:.0f}%)")
    print(f"      - Average age: {avg_age:.1f} years")
    if count > 0:
        top_3_count = int(np.minimum(3, count))
        top_3 = asset_metrics[asset_metrics['category'] == category].nlargest(top_3_count, 'performance_score')
        print(f"      - Top priorities: {', '.join(top_3['substation_id'].tolist())}")

print("\n4. MODERN vs AGING INFRASTRUCTURE:")
modern_threshold = 10
modern_assets = asset_metrics[asset_metrics['asset_age_years'] < modern_threshold]
aging_assets = asset_metrics[asset_metrics['asset_age_years'] >= modern_threshold]

if len(modern_assets) > 0:
    print(f"\n   Modern (<{modern_threshold}yr): {len(modern_assets)} assets")
    print(f"      - Avg incidents: {modern_assets['incident_count'].mean():.1f}")
    print(f"      - Avg duration: {modern_assets['duration_minutes'].mean():.0f} min")
    print(f"      - Avg performance score: {modern_assets['performance_score'].mean():.1f}")

if len(aging_assets) > 0:
    print(f"\n   Aging (≥{modern_threshold}yr): {len(aging_assets)} assets")
    print(f"      - Avg incidents: {aging_assets['incident_count'].mean():.1f}")
    print(f"      - Avg duration: {aging_assets['duration_minutes'].mean():.0f} min")
    print(f"      - Avg performance score: {aging_assets['performance_score'].mean():.1f}")

if len(modern_assets) > 0 and len(aging_assets) > 0:
    performance_ratio = aging_assets['performance_score'].mean() / modern_assets['performance_score'].mean()
    print(f"\n   → Aging assets are {performance_ratio:.1f}x worse performers")

print("\n" + "="*80)
print("🎯 KEY INSIGHTS:")
print("="*80)
print(f"• {high_risk_pct:.0f}% of portfolio is in high-risk age categories (15+ years)")
print(f"• {pct_assets_for_80:.0f}% of assets cause 80% of total risk (Pareto principle)")
replace_now_count = (asset_metrics['category'] == 'Replace Now').sum()
print(f"• {replace_now_count} assets need immediate replacement (old + problematic)")
investigate_count = (asset_metrics['category'] == 'Investigate').sum()
print(f"• {investigate_count} young assets underperforming (operational issues, not age)")
if len(modern_assets) > 0 and len(aging_assets) > 0:
    print(f"• Aging infrastructure ({modern_threshold}+ yr) performs {performance_ratio:.1f}x worse than modern")
print("="*80)

## ⛓️ Question 9: The Cascade Effect

**Analyzing Systemic Failures:**
- Multi-incident days (when one failure triggers others)
- Geographic clustering
- Domino effect patterns
- Grid stress concentration

**What we'll discover:** When failures spread like wildfires

In [0]:
# Analyze multi-incident patterns
operations_pd = df_operations.toPandas()
operations_pd['event_date'] = pd.to_datetime(operations_pd['timestamp']).dt.date

# Daily incident aggregation
daily_incidents = operations_pd.groupby('event_date').agg({
    'event_id': 'count',
    'region_name': lambda x: x.nunique(),
    'substation_id': lambda x: x.nunique(),
    'duration_minutes': 'sum',
    'affected_customers': 'sum',
    'severity': lambda x: (x == 'critical').sum()
}).rename(columns={
    'event_id': 'incident_count',
    'region_name': 'regions_affected',
    'substation_id': 'substations_affected',
    'duration_minutes': 'total_duration',
    'affected_customers': 'total_customers',
    'severity': 'critical_count'
}).reset_index()

# Identify cascade days (multiple incidents on same day)
daily_incidents['is_cascade'] = daily_incidents['incident_count'] > 1

# Regional clustering on same day
regional_cascade = operations_pd.groupby(['event_date', 'region_name']).agg({
    'event_id': 'count',
    'timestamp': lambda x: (pd.to_datetime(x).max() - pd.to_datetime(x).min()).total_seconds() / 3600
}).rename(columns={
    'event_id': 'regional_incidents',
    'timestamp': 'time_span_hours'
}).reset_index()

regional_cascade_multi = regional_cascade[regional_cascade['regional_incidents'] > 1]

# Create 2x2 visualization with CHEERFUL COLORS
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# Panel 1: Cascade Days Timeline
ax1 = axes[0, 0]

daily_incidents_sorted = daily_incidents.sort_values('event_date')
cascade_days = daily_incidents_sorted[daily_incidents_sorted['is_cascade']]
single_days = daily_incidents_sorted[~daily_incidents_sorted['is_cascade']]

# Plot single-incident days
ax1.scatter(single_days['event_date'], single_days['incident_count'],
           s=100, color='#B4E7CE', alpha=0.5, edgecolors='white', linewidth=1.5, label='Single Incident')

# Plot cascade days with size proportional to total customers affected
scatter = ax1.scatter(cascade_days['event_date'], cascade_days['incident_count'],
                     s=cascade_days['total_customers']/30, c=cascade_days['critical_count'],
                     cmap='YlOrRd', alpha=0.8, edgecolors='white', linewidth=2, label='Cascade Event')

ax1.set_xlabel('Date (January 2024)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Incidents per Day', fontsize=12, fontweight='bold')
ax1.set_title('Cascade Days: When Failures Cluster', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.tick_params(axis='x', rotation=45)
ax1.legend(loc='upper left', fontsize=10)

# Add colorbar
cbar1 = plt.colorbar(scatter, ax=ax1)
cbar1.set_label('Critical Incidents', fontsize=10, fontweight='bold')

# Highlight worst cascade day
worst_day = cascade_days.loc[cascade_days['total_customers'].idxmax()]
ax1.annotate(f"Worst day:\n{worst_day['incident_count']} incidents\n{int(worst_day['total_customers']):,} customers",
            xy=(worst_day['event_date'], worst_day['incident_count']),
            xytext=(worst_day['event_date'], worst_day['incident_count'] + 0.7),
            fontsize=10, fontweight='bold', color='#FF6B9D',
            bbox=dict(boxstyle='round', facecolor='#FFE5F1', edgecolor='#FF6B9D', linewidth=2),
            arrowprops=dict(arrowstyle='->', lw=2, color='#FF6B9D'))

# Panel 2: Cascade Severity Distribution
ax2 = axes[0, 1]

# Categorize days by incident count
cascade_days_copy = cascade_days.copy()
cascade_days_copy['cascade_severity'] = cascade_days_copy['incident_count'].apply(
    lambda x: '2 incidents' if x == 2 else '3 incidents' if x == 3 else '4+ incidents'
)

cascade_severity_counts = cascade_days_copy['cascade_severity'].value_counts().reindex(
    ['2 incidents', '3 incidents', '4+ incidents'], fill_value=0
)

colors_cascade = ['#FFD93D', '#FFB86C', '#FF9FF3']
bars = ax2.bar(range(len(cascade_severity_counts)), cascade_severity_counts.values,
              color=colors_cascade, edgecolor='white', linewidth=2)

ax2.set_xticks(range(len(cascade_severity_counts)))
ax2.set_xticklabels(cascade_severity_counts.index, fontsize=11)
ax2.set_ylabel('Number of Days', fontsize=12, fontweight='bold')
ax2.set_title('Cascade Severity Distribution', fontsize=14, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

# Add value labels and percentages
total_cascade_days = len(cascade_days)
for bar, val in zip(bars, cascade_severity_counts.values):
    if val > 0:
        pct = (val / total_cascade_days) * 100
        ax2.text(bar.get_x() + bar.get_width()/2., val + 0.1,
                f'{int(val)}\n({pct:.0f}%)',
                ha='center', va='bottom', fontsize=11, fontweight='bold')

# Add statistics box
stats_text = f"Total cascade days: {total_cascade_days}\nTotal days with incidents: {len(daily_incidents)}\nCascade rate: {total_cascade_days/len(daily_incidents)*100:.0f}%"
ax2.text(0.98, 0.97, stats_text, transform=ax2.transAxes,
        fontsize=10, verticalalignment='top', horizontalalignment='right',
        bbox=dict(boxstyle='round', facecolor='#FFF9E6', edgecolor='#FFD93D', linewidth=2))

# Panel 3: Regional Clustering Heatmap
ax3 = axes[1, 0]

# Create pivot table for regional clustering
regional_pivot = regional_cascade.pivot_table(
    index='region_name',
    columns=pd.to_datetime(regional_cascade['event_date']).dt.day,
    values='regional_incidents',
    fill_value=0
)

# Plot heatmap
im = ax3.imshow(regional_pivot.values, cmap='YlOrRd', aspect='auto', interpolation='nearest')
ax3.set_xticks(range(len(regional_pivot.columns)))
ax3.set_xticklabels(regional_pivot.columns, fontsize=9)
ax3.set_yticks(range(len(regional_pivot.index)))
ax3.set_yticklabels(regional_pivot.index, fontsize=11)
ax3.set_xlabel('Day of Month (January 2024)', fontsize=12, fontweight='bold')
ax3.set_ylabel('Region', fontsize=12, fontweight='bold')
ax3.set_title('Regional Clustering: Geographic Cascade Patterns', fontsize=14, fontweight='bold')

# Add colorbar
cbar3 = plt.colorbar(im, ax=ax3)
cbar3.set_label('Incidents per Region-Day', fontsize=10, fontweight='bold')

# Annotate multi-incident cells
for i in range(len(regional_pivot.index)):
    for j in range(len(regional_pivot.columns)):
        value = regional_pivot.values[i, j]
        if value > 1:
            ax3.text(j, i, int(value), ha='center', va='center',
                    color='white' if value > 1.5 else 'black',
                    fontsize=11, fontweight='bold')

# Panel 4: Cascade Impact Analysis
ax4 = axes[1, 1]

# Compare cascade vs single-incident days
metrics = ['total_customers', 'total_duration', 'regions_affected']
metric_labels = ['Customers\nAffected', 'Total Duration\n(minutes)', 'Regions\nAffected']

cascade_means = [cascade_days[m].mean() for m in metrics]
single_means = [single_days[m].mean() for m in metrics]

# Normalize for comparison (index to single-incident days)
ratios = [c / s if s > 0 else 0 for c, s in zip(cascade_means, single_means)]

x = np.arange(len(metrics))
width = 0.35

bars1 = ax4.bar(x - width/2, single_means, width, label='Single-Incident Days',
               color='#B4E7CE', edgecolor='white', linewidth=2)
bars2 = ax4.bar(x + width/2, cascade_means, width, label='Cascade Days',
               color='#FF9FF3', edgecolor='white', linewidth=2)

ax4.set_ylabel('Average Value', fontsize=12, fontweight='bold')
ax4.set_xlabel('Impact Metric', fontsize=12, fontweight='bold')
ax4.set_title('Cascade Days: Amplified Impact', fontsize=14, fontweight='bold')
ax4.set_xticks(x)
ax4.set_xticklabels(metric_labels, fontsize=11)
ax4.legend(loc='upper left', fontsize=10)
ax4.grid(axis='y', alpha=0.3)

# Add ratio labels
for i, (bar, ratio) in enumerate(zip(bars2, ratios)):
    height = bar.get_height()
    ax4.text(bar.get_x() + bar.get_width()/2., height + height*0.05,
            f'{ratio:.1f}x',
            ha='center', va='bottom', fontsize=11, fontweight='bold', color='#FF9FF3')

plt.suptitle('The Cascade Effect: When Failures Trigger Failures', fontsize=17, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("CASCADE EFFECT ANALYSIS")
print("="*80)

print("\n1. CASCADE FREQUENCY:")
print(f"   Total days with incidents: {len(daily_incidents)}")
print(f"   Cascade days (multiple incidents): {total_cascade_days}")
print(f"   Cascade rate: {total_cascade_days/len(daily_incidents)*100:.0f}%")
print(f"\n   Single-incident days: {len(single_days)}")
for severity, count in cascade_severity_counts.items():
    if count > 0:
        print(f"   {severity}: {count} days")

print("\n2. WORST CASCADE DAYS:")
top_cascade = cascade_days.nlargest(5, 'total_customers')[['event_date', 'incident_count', 'regions_affected', 'total_customers', 'critical_count']]
for idx, (_, day) in enumerate(top_cascade.iterrows(), 1):
    print(f"\n   #{idx}: {day['event_date']}")
    print(f"      - Incidents: {day['incident_count']}")
    print(f"      - Regions: {day['regions_affected']}")
    print(f"      - Customers: {int(day['total_customers']):,}")
    print(f"      - Critical incidents: {day['critical_count']}")

print("\n3. REGIONAL CLUSTERING:")
print(f"   Total region-days with multiple incidents: {len(regional_cascade_multi)}")
if len(regional_cascade_multi) > 0:
    print("\n   Worst regional clusters:")
    worst_clusters = regional_cascade_multi.nlargest(5, 'regional_incidents')[['event_date', 'region_name', 'regional_incidents', 'time_span_hours']]
    for idx, (_, cluster) in enumerate(worst_clusters.iterrows(), 1):
        print(f"      #{idx}: {cluster['region_name']} on {cluster['event_date']}")
        print(f"         - Incidents: {cluster['regional_incidents']}")
        print(f"         - Time span: {cluster['time_span_hours']:.1f} hours")

print("\n4. CASCADE IMPACT AMPLIFICATION:")
for metric, label, ratio, cascade_val, single_val in zip(
    metrics, ['Customers affected', 'Total duration (min)', 'Regions affected'],
    ratios, cascade_means, single_means
):
    print(f"\n   {label}:")
    print(f"      - Single-incident days: {single_val:.0f}")
    print(f"      - Cascade days: {cascade_val:.0f}")
    print(f"      - Amplification: {ratio:.1f}x")

print("\n5. GEOGRAPHIC SPREAD:")
regions_in_cascade = cascade_days['regions_affected'].mean()
print(f"   Average regions affected per cascade day: {regions_in_cascade:.1f}")
multi_region_cascades = (cascade_days['regions_affected'] > 1).sum()
print(f"   Multi-region cascades: {multi_region_cascades} ({multi_region_cascades/len(cascade_days)*100:.0f}%)")

print("\n" + "="*80)
print("🎯 KEY INSIGHTS:")
print("="*80)
print(f"• {total_cascade_days/len(daily_incidents)*100:.0f}% of incident days involve multiple failures (cascade effect)")
print(f"• Cascade days affect {ratios[0]:.1f}x more customers than single-incident days")
print(f"• {multi_region_cascades} cascade days spread across multiple regions (systemic risk)")
if len(regional_cascade_multi) > 0:
    most_clustered = regional_cascade_multi.groupby('region_name')['regional_incidents'].sum().idxmax()
    print(f"• {most_clustered} experiences the most regional clustering")
print(f"• When one asset fails, it triggers {cascade_days['incident_count'].mean():.1f} incidents on average")
print("="*80)

## 💰 Question 10: Market Price vs Grid Stress

**Economic Impact Analysis:**
- Do failures happen when electricity is most expensive?
- Lost revenue calculation
- Price volatility during incidents
- Economic opportunity cost

**What we'll discover:** The financial pain of poor timing

In [0]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Load and prepare market price data
market_prices_pd = df_market_prices.toPandas()
market_prices_pd['report_day'] = pd.to_datetime(market_prices_pd['report_day'])
market_prices_pd['timestamp'] = pd.to_datetime(market_prices_pd['timestamp'])

# Map region_normalized (country codes) to country names
region_to_country = {
    'DK': 'Denmark',
    'FI': 'Finland',
    'SE': 'Sweden',
    'NO': 'Norway'
}
market_prices_pd['country'] = market_prices_pd['region_normalized'].map(region_to_country)

# Prepare operations data
operations_pd = df_operations.toPandas()
operations_pd['timestamp'] = pd.to_datetime(operations_pd['timestamp'])
operations_pd['event_date'] = operations_pd['timestamp'].dt.date

# Map regions to countries for price matching
region_to_country_ops = {
    'Copenhagen': 'Denmark',
    'Helsinki': 'Finland',
    'Turku': 'Finland',
    'Gothenburg': 'Sweden',
    'Malmo': 'Sweden',
    'Tampere': 'Finland'
}

operations_pd['country'] = operations_pd['region_name'].map(region_to_country_ops)

# Match incidents to market prices (same day + country)
incidents_with_prices = operations_pd.merge(
    market_prices_pd[['report_day', 'country', 'price_eur_mwh']],
    left_on=['event_date', 'country'],
    right_on=[market_prices_pd['report_day'].dt.date, 'country'],
    how='left'
)

# Calculate daily average prices by country
daily_avg_prices = market_prices_pd.groupby([market_prices_pd['report_day'].dt.date, 'country'])['price_eur_mwh'].mean().reset_index()
daily_avg_prices.columns = ['date', 'country', 'daily_avg_price']

# Calculate economic impact (lost revenue opportunity)
# Assumption: Average household consumption is 1 kWh/hour
incidents_with_prices['energy_not_delivered_mwh'] = (incidents_with_prices['affected_customers'] * incidents_with_prices['duration_minutes']) / (1000 * 60)
incidents_with_prices['lost_revenue_eur'] = incidents_with_prices['energy_not_delivered_mwh'] * incidents_with_prices['price_eur_mwh']

# Create 2x2 visualization
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# Panel 1: Incident Timing vs Price Levels
ax1 = axes[0, 0]

# Define price quartiles
price_quartiles = market_prices_pd['price_eur_mwh'].quantile([0.25, 0.5, 0.75])
incidents_with_prices['price_category'] = pd.cut(
    incidents_with_prices['price_eur_mwh'],
    bins=[0, price_quartiles[0.25], price_quartiles[0.5], price_quartiles[0.75], 1000],
    labels=['Low Price', 'Medium-Low', 'Medium-High', 'High Price']
)

price_dist = incidents_with_prices['price_category'].value_counts().reindex(
    ['Low Price', 'Medium-Low', 'Medium-High', 'High Price'], fill_value=0
)

colors_price = ['#27AE60', '#F39C12', '#E67E22', '#E74C3C']
bars = ax1.bar(range(len(price_dist)), price_dist.values, color=colors_price, edgecolor='black', linewidth=2)

ax1.set_xticks(range(len(price_dist)))
ax1.set_xticklabels(price_dist.index, fontsize=11)
ax1.set_ylabel('Number of Incidents', fontsize=12, fontweight='bold')
ax1.set_title('When Do Failures Happen? Price Context', fontsize=14, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# Add value labels
for bar, val in zip(bars, price_dist.values):
    if val > 0:
        pct = (val / price_dist.sum()) * 100
        ax1.text(bar.get_x() + bar.get_width()/2., val + 0.1,
                f'{int(val)}\n({pct:.0f}%)',
                ha='center', va='bottom', fontsize=11, fontweight='bold')

# Add expected distribution line (25% each quartile)
ax1.axhline(y=len(incidents_with_prices)/4, color='#2C3E50', linestyle='--', linewidth=2, label='Expected (25% each)')
ax1.legend(loc='upper right', fontsize=10)

# Panel 2: Lost Revenue by Incident
ax2 = axes[0, 1]

# Top 10 most costly incidents
top_costly = incidents_with_prices.nlargest(10, 'lost_revenue_eur')

y_pos = np.arange(len(top_costly))
colors_revenue = plt.cm.Reds(np.linspace(0.4, 0.9, len(top_costly)))

bars = ax2.barh(y_pos, top_costly['lost_revenue_eur'], color=colors_revenue, edgecolor='black', linewidth=2)
ax2.set_yticks(y_pos)
ax2.set_yticklabels([f"{row['substation_id']}\n{row['region_name']}" for _, row in top_costly.iterrows()], fontsize=9)
ax2.set_xlabel('Lost Revenue (EUR)', fontsize=12, fontweight='bold')
ax2.set_title('Top 10 Most Costly Incidents', fontsize=14, fontweight='bold')
ax2.grid(axis='x', alpha=0.3)

# Add value labels
for i, (bar, val) in enumerate(zip(bars, top_costly['lost_revenue_eur'])):
    ax2.text(val + 50, bar.get_y() + bar.get_height()/2,
            f'€{val:.0f}',
            va='center', fontsize=9, fontweight='bold')

# Panel 3: Price Timeline with Incident Overlay
ax3 = axes[1, 0]

# Plot price trends by country
for country in ['Denmark', 'Finland', 'Sweden']:
    country_prices = daily_avg_prices[daily_avg_prices['country'] == country].sort_values('date')
    ax3.plot(pd.to_datetime(country_prices['date']), country_prices['daily_avg_price'],
            marker='o', linewidth=2, markersize=4, label=country, alpha=0.7)

# Overlay incidents as vertical spans
for _, incident in incidents_with_prices.iterrows():
    incident_date = pd.to_datetime(incident['event_date'])
    if incident['severity'] == 'critical':
        ax3.axvline(x=incident_date, color='#E74C3C', alpha=0.3, linewidth=2, linestyle='--')

ax3.set_xlabel('Date (January 2024)', fontsize=12, fontweight='bold')
ax3.set_ylabel('Avg Spot Price (EUR/MWh)', fontsize=12, fontweight='bold')
ax3.set_title('Price Volatility & Incident Timing', fontsize=14, fontweight='bold')
ax3.legend(loc='upper right', fontsize=10)
ax3.grid(True, alpha=0.3)
ax3.tick_params(axis='x', rotation=45)

# Add annotation
ax3.text(0.02, 0.98, 'Red dashes = Critical incidents',
        transform=ax3.transAxes, fontsize=10, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='#FFE6E6', edgecolor='#E74C3C', linewidth=2))

# Panel 4: Economic Impact Summary
ax4 = axes[1, 1]

# Calculate metrics by country
economic_by_country = incidents_with_prices.groupby('country').agg({
    'lost_revenue_eur': 'sum',
    'event_id': 'count',
    'affected_customers': 'sum'
}).rename(columns={'event_id': 'incident_count'}).sort_values('lost_revenue_eur', ascending=True)

# Create stacked bar chart showing breakdown
y_pos = np.arange(len(economic_by_country))
colors_country = ['#3498DB', '#E67E22', '#9B59B6']

bars = ax4.barh(y_pos, economic_by_country['lost_revenue_eur'], color=colors_country, edgecolor='black', linewidth=2)
ax4.set_yticks(y_pos)
ax4.set_yticklabels(economic_by_country.index, fontsize=11)
ax4.set_xlabel('Total Lost Revenue (EUR)', fontsize=12, fontweight='bold')
ax4.set_title('Economic Impact by Country', fontsize=14, fontweight='bold')
ax4.grid(axis='x', alpha=0.3)

# Add value labels with details
for i, (bar, idx) in enumerate(zip(bars, economic_by_country.index)):
    val = economic_by_country.loc[idx, 'lost_revenue_eur']
    incidents = int(economic_by_country.loc[idx, 'incident_count'])
    customers = int(economic_by_country.loc[idx, 'affected_customers'])
    ax4.text(val + 100, bar.get_y() + bar.get_height()/2,
            f'€{val:.0f}\n({incidents} incidents)',
            va='center', fontsize=10, fontweight='bold')

plt.suptitle('Market Price vs Grid Stress: The Economic Cost of Bad Timing', fontsize=17, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("MARKET PRICE vs GRID STRESS ANALYSIS")
print("="*80)

print("\n1. INCIDENT TIMING vs PRICE LEVELS:")
for category, count in price_dist.items():
    pct = (count / price_dist.sum()) * 100
    print(f"   {category}: {count} incidents ({pct:.0f}%)")

expected_pct = 25
high_price_pct = (price_dist['High Price'] / price_dist.sum()) * 100
if high_price_pct > expected_pct:
    print(f"\n   ⚠️ {high_price_pct:.0f}% of incidents during high-price periods (expected: {expected_pct}%)")
    print(f"   → Failures are {high_price_pct/expected_pct:.1f}x more likely during expensive hours")
else:
    print(f"\n   ✅ Incidents are evenly distributed across price levels")

print("\n2. TOP 5 MOST COSTLY INCIDENTS:")
for idx, (_, incident) in enumerate(top_costly.head(5).iterrows(), 1):
    print(f"\n   #{idx}: {incident['substation_id']} ({incident['region_name']})")
    print(f"      - Lost revenue: €{incident['lost_revenue_eur']:.0f}")
    print(f"      - Spot price: €{incident['price_eur_mwh']:.2f}/MWh")
    print(f"      - Energy not delivered: {incident['energy_not_delivered_mwh']:.1f} MWh")
    print(f"      - Duration: {incident['duration_minutes']:.0f} min")
    print(f"      - Customers: {int(incident['affected_customers']):,}")

print("\n3. ECONOMIC IMPACT BY COUNTRY:")
total_lost_revenue = incidents_with_prices['lost_revenue_eur'].sum()
for country, data in economic_by_country.iterrows():
    pct = (data['lost_revenue_eur'] / total_lost_revenue) * 100
    avg_per_incident = data['lost_revenue_eur'] / data['incident_count']
    print(f"\n   {country}:")
    print(f"      - Total lost revenue: €{data['lost_revenue_eur']:.0f} ({pct:.0f}%)")
    print(f"      - Incidents: {int(data['incident_count'])}")
    print(f"      - Avg loss per incident: €{avg_per_incident:.0f}")

print("\n4. PRICE VOLATILITY DURING INCIDENTS:")
incident_days = set(incidents_with_prices['event_date'])
prices_on_incident_days = daily_avg_prices[daily_avg_prices['date'].isin(incident_days)]['daily_avg_price']
prices_on_normal_days = daily_avg_prices[~daily_avg_prices['date'].isin(incident_days)]['daily_avg_price']

print(f"   Average price on incident days: €{prices_on_incident_days.mean():.2f}/MWh")
print(f"   Average price on normal days: €{prices_on_normal_days.mean():.2f}/MWh")
price_diff = ((prices_on_incident_days.mean() / prices_on_normal_days.mean()) - 1) * 100

if price_diff > 10:
    print(f"\n   ⚠️ Prices are {abs(price_diff):.0f}% HIGHER on incident days")
    print(f"   → Bad timing: failures happening when electricity is most valuable")
elif price_diff < -10:
    print(f"\n   ✅ Prices are {abs(price_diff):.0f}% LOWER on incident days")
    print(f"   → Good timing: failures happening when electricity is less valuable")
else:
    print(f"\n   → Prices similar on incident vs normal days ({price_diff:+.0f}%)")

print("\n5. TOTAL ECONOMIC IMPACT:")
print(f"   Total lost revenue: €{total_lost_revenue:.0f}")
print(f"   Average per incident: €{total_lost_revenue / len(incidents_with_prices):.0f}")
print(f"   Most expensive hour: {incidents_with_prices.loc[incidents_with_prices['price_eur_mwh'].idxmax(), 'price_eur_mwh']:.2f} EUR/MWh")
print(f"   Cheapest hour: {incidents_with_prices.loc[incidents_with_prices['price_eur_mwh'].idxmin(), 'price_eur_mwh']:.2f} EUR/MWh")

print("\n" + "="*80)